In [ ]:
import pandas as pd
import plotly.graph_objects as go
import numpy as np

# Read the categorization data from 'Experiment Tracker.xlsx'
df = pd.read_excel('Experiment Tracker.xlsx', sheet_name=['T30 categorization', 'T60 categorization'])

# Categorization at time=30
group_10 = set(df['T30 categorization']['Group_10'].dropna())
group_20 = set(df['T30 categorization']['Group_20'].dropna())
group_30 = set(df['T30 categorization']['Group_30'].dropna())

# Categorization at time=60
full_occlusion = set(df['T60 categorization']['Full_Occlusion'].dropna())
partial_occlusion = set(df['T60 categorization']['Partial_Occlusion'].dropna())
no_occlusion = set(df['T60 categorization']['No_Occlusion'].dropna())

# Find overlapping subjects for each combination at time=60
hemorrhage_10_full_occlusion = group_10.intersection(full_occlusion)
hemorrhage_10_partial_occlusion = group_10.intersection(partial_occlusion)
hemorrhage_10_no_occlusion = group_10.intersection(no_occlusion)

hemorrhage_20_full_occlusion = group_20.intersection(full_occlusion)
hemorrhage_20_partial_occlusion = group_20.intersection(partial_occlusion)
hemorrhage_20_no_occlusion = group_20.intersection(no_occlusion)

hemorrhage_30_full_occlusion = group_30.intersection(full_occlusion)
hemorrhage_30_partial_occlusion = group_30.intersection(partial_occlusion)
hemorrhage_30_no_occlusion = group_30.intersection(no_occlusion)

# Now load the Excel file containing the main data (processed data)
file_path = "Processed_Data_with_Vasopressin.xlsx"
xls = pd.ExcelFile(file_path)

# Filter sheets that start with "ERNE" and not "ERDE"
erne_sheets = [sheet for sheet in xls.sheet_names if sheet.startswith("ERNE")]

# Dictionary to store dynamically generated variables
data_dict = {}

# Loop through the filtered sheets and store them in the dictionary
for sheet in erne_sheets:
    new_sheet_name = sheet.replace("-", "")
    df = xls.parse(sheet)
    required_columns = ["Time", "Cumulative Vasopressin", "Plasmalyte", "Norepi"]
    df = df[required_columns]
    # Remove rows where 'Time' is greater than 240
    df = df[df["Time"] <= 240]
    data_dict[new_sheet_name] = df

# Define a function to manually calculate standard deviation for each group combination
def calculate_mean_and_std(group_data, time_col='Time', value_col='Cumulative Vasopressin'):
    """
    Calculate the mean and standard deviation for a given group.
    """
    group_values = []
    times = []
    
    # Aggregating data for each time point
    for group in group_data:
        if group in data_dict:
            df = data_dict[group]
            group_times = df[time_col].unique()  # Time points
            for time in group_times:
                group_at_time = df[df[time_col] == time]
                group_values.extend(group_at_time[value_col].tolist())
                times.extend([time] * len(group_at_time[value_col]))

    # Calculate mean and standard deviation for each time point
    unique_times = sorted(set(times))
    means = []
    std_devs = []
    
    for time in unique_times:
        values_at_time = [group_values[i] for i, t in enumerate(times) if t == time]
        means.append(np.mean(values_at_time))
        std_devs.append(np.std(values_at_time))  # Standard deviation

    return unique_times, means, std_devs

# Create combined groups (time=60 categorization)
group_combinations = {
    "10% Hemorrhage + Full Occlusion": hemorrhage_10_full_occlusion,
    "10% Hemorrhage + Partial Occlusion": hemorrhage_10_partial_occlusion,
    "10% Hemorrhage + No Occlusion": hemorrhage_10_no_occlusion,
    "20% Hemorrhage + Full Occlusion": hemorrhage_20_full_occlusion,
    "20% Hemorrhage + Partial Occlusion": hemorrhage_20_partial_occlusion,
    "20% Hemorrhage + No Occlusion": hemorrhage_20_no_occlusion,
    "30% Hemorrhage + Full Occlusion": hemorrhage_30_full_occlusion,
    "30% Hemorrhage + Partial Occlusion": hemorrhage_30_partial_occlusion,
    "30% Hemorrhage + No Occlusion": hemorrhage_30_no_occlusion,
}

# Prepare data for plotting (for 'Cumulative Vasopressin')
traces_vasopressin = []
for group_name, group_data in group_combinations.items():
    if group_data:  # Only calculate if there is data
        times, means, std_devs = calculate_mean_and_std(group_data, value_col='Cumulative Vasopressin')
        
        # Add to the plot
        traces_vasopressin.append(go.Scatter(
            x=times, y=means,
            mode='lines+markers',
            name=group_name,
            error_y=dict(type='data', array=std_devs, visible=True),  # Error bars (Standard Deviation)
            line=dict(width=2),
            marker=dict(size=6)
        ))

# Prepare data for plotting (for 'Plasmalyte')
traces_plasmalyte = []
for group_name, group_data in group_combinations.items():
    if group_data:  # Only calculate if there is data
        times, means, std_devs = calculate_mean_and_std(group_data, value_col='Plasmalyte')
        
        # Add to the plot
        traces_plasmalyte.append(go.Scatter(
            x=times, y=means,
            mode='lines+markers',
            name=group_name,
            error_y=dict(type='data', array=std_devs, visible=True),  # Error bars (Standard Deviation)
            line=dict(width=2),
            marker=dict(size=6)
        ))

# Prepare data for plotting (for 'Norepi')
traces_norepi = []
for group_name, group_data in group_combinations.items():
    if group_data:  # Only calculate if there is data
        times, means, std_devs = calculate_mean_and_std(group_data, value_col='Norepi')
        
        # Add to the plot
        traces_norepi.append(go.Scatter(
            x=times, y=means,
            mode='lines+markers',
            name=group_name,
            error_y=dict(type='data', array=std_devs, visible=True),  # Error bars (Standard Deviation)
            line=dict(width=2),
            marker=dict(size=6)
        ))

# Create layout for the graphs
layout = go.Layout(
    title="Pooled Averages of Parameters for Different Group Combinations",
    xaxis=dict(title='Time (minutes)'),
    yaxis=dict(title='Value'),
    showlegend=True
)

# Create the figure for Total Vasopressin
fig_vasopressin = go.Figure(data=traces_vasopressin, layout=layout)
fig_vasopressin.update_layout(title='Cumulative Vasopressin with Error Bars')

# Create the figure for Plasmalyte
fig_plasmalyte = go.Figure(data=traces_plasmalyte, layout=layout)
fig_plasmalyte.update_layout(title='Plasmalyte with Error Bars')

# Create the figure for Norepi
fig_norepi = go.Figure(data=traces_norepi, layout=layout)
fig_norepi.update_layout(title='Norepi with Error Bars')

# Show the figures
fig_vasopressin.show()
fig_plasmalyte.show()
fig_norepi.show()


In [ ]:
import os
import pandas as pd
import plotly.graph_objects as go
import numpy as np

# Create the "drug concentrations" directory if it doesn't exist
output_dir = "drug concentrations"
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

# Read the categorization data from 'Experiment Tracker.xlsx'
df = pd.read_excel('Experiment Tracker.xlsx', sheet_name=['T30 categorization', 'T60 categorization'])

# Categorization at time=30
group_10 = set(df['T30 categorization']['Group_10'].dropna())
group_20 = set(df['T30 categorization']['Group_20'].dropna())
group_30 = set(df['T30 categorization']['Group_30'].dropna())

# Categorization at time=60
full_occlusion = set(df['T60 categorization']['Full_Occlusion'].dropna())
partial_occlusion = set(df['T60 categorization']['Partial_Occlusion'].dropna())
no_occlusion = set(df['T60 categorization']['No_Occlusion'].dropna())

# Find overlapping subjects for each combination at time=60
hemorrhage_10_full_occlusion = group_10.intersection(full_occlusion)
hemorrhage_10_partial_occlusion = group_10.intersection(partial_occlusion)
hemorrhage_10_no_occlusion = group_10.intersection(no_occlusion)

hemorrhage_20_full_occlusion = group_20.intersection(full_occlusion)
hemorrhage_20_partial_occlusion = group_20.intersection(partial_occlusion)
hemorrhage_20_no_occlusion = group_20.intersection(no_occlusion)

hemorrhage_30_full_occlusion = group_30.intersection(full_occlusion)
hemorrhage_30_partial_occlusion = group_30.intersection(partial_occlusion)
hemorrhage_30_no_occlusion = group_30.intersection(no_occlusion)

# Now load the Excel file containing the main data (processed data)
file_path = "Processed_Data_with_Vasopressin.xlsx"
xls = pd.ExcelFile(file_path)

# Filter sheets that start with "ERNE" and not "ERDE"
erne_sheets = [sheet for sheet in xls.sheet_names if sheet.startswith("ERNE")]

# Dictionary to store dynamically generated variables
data_dict = {}

# Loop through the filtered sheets and store them in the dictionary
for sheet in erne_sheets:
    new_sheet_name = sheet.replace("-", "")
    df = xls.parse(sheet)
    required_columns = ["Time", "Cumulative Vasopressin", "Plasmalyte", "Norepi"]
    df = df[required_columns]
    # Remove rows where 'Time' is greater than 240
    df = df[df["Time"] <= 240]
    data_dict[new_sheet_name] = df

# Define a function to manually calculate standard deviation for each group combination, keeping only specified time points
def calculate_mean_and_std(group_data, time_col='Time', value_col='Cumulative Vasopressin', time_points=None):
    """
    Calculate the mean and standard deviation for a given group, only for specified time points.
    """
    group_values = []
    times = []
    
    # Aggregating data for each time point
    for group in group_data:
        if group in data_dict:
            df = data_dict[group]
            group_times = df[time_col].unique()  # Time points
            # Filter to keep only the specified time points, if any
            if time_points:
                group_times = [time for time in group_times if time in time_points]
            for time in group_times:
                group_at_time = df[df[time_col] == time]
                group_values.extend(group_at_time[value_col].tolist())
                times.extend([time] * len(group_at_time[value_col]))

    # Calculate mean and standard deviation for each time point
    unique_times = sorted(set(times))
    means = []
    std_devs = []
    
    for time in unique_times:
        values_at_time = [group_values[i] for i, t in enumerate(times) if t == time]
        means.append(np.mean(values_at_time))
        std_devs.append(np.std(values_at_time))  # Standard deviation

    return unique_times, means, std_devs

# Define your list of time points to keep (as you provided)
time_points_to_keep = [0,5,10,15,20,25,30,60,65,75,85,120,180,240]

# Create combined groups (time=60 categorization)
group_combinations = {
    "10% Hemorrhage + Full Occlusion": hemorrhage_10_full_occlusion,
    "10% Hemorrhage + Partial Occlusion": hemorrhage_10_partial_occlusion,
    "10% Hemorrhage + No Occlusion": hemorrhage_10_no_occlusion,
    "20% Hemorrhage + Full Occlusion": hemorrhage_20_full_occlusion,
    "20% Hemorrhage + Partial Occlusion": hemorrhage_20_partial_occlusion,
    "20% Hemorrhage + No Occlusion": hemorrhage_20_no_occlusion,
    "30% Hemorrhage + Full Occlusion": hemorrhage_30_full_occlusion,
    "30% Hemorrhage + Partial Occlusion": hemorrhage_30_partial_occlusion,
    "30% Hemorrhage + No Occlusion": hemorrhage_30_no_occlusion,
}

# Prepare data for plotting (for 'Cumulative Vasopressin')
traces_vasopressin = []
for group_name, group_data in group_combinations.items():
    if group_data:  # Only calculate if there is data
        times, means, std_devs = calculate_mean_and_std(group_data, value_col='Cumulative Vasopressin', time_points=time_points_to_keep)
        
        # Add to the plot
        traces_vasopressin.append(go.Scatter(
            x=times, y=means,
            mode='lines+markers',
            name=group_name,
            error_y=dict(type='data', array=std_devs, visible=True),  # Error bars (Standard Deviation)
            line=dict(width=2),
            marker=dict(size=6)
        ))

# Prepare data for plotting (for 'Plasmalyte')
traces_plasmalyte = []
for group_name, group_data in group_combinations.items():
    if group_data:  # Only calculate if there is data
        times, means, std_devs = calculate_mean_and_std(group_data, value_col='Plasmalyte', time_points=time_points_to_keep)
        
        # Add to the plot
        traces_plasmalyte.append(go.Scatter(
            x=times, y=means,
            mode='lines+markers',
            name=group_name,
            error_y=dict(type='data', array=std_devs, visible=True),  # Error bars (Standard Deviation)
            line=dict(width=2),
            marker=dict(size=6)
        ))

# Prepare data for plotting (for 'Norepi')
traces_norepi = []
for group_name, group_data in group_combinations.items():
    if group_data:  # Only calculate if there is data
        times, means, std_devs = calculate_mean_and_std(group_data, value_col='Norepi', time_points=time_points_to_keep)
        
        # Add to the plot
        traces_norepi.append(go.Scatter(
            x=times, y=means,
            mode='lines+markers',
            name=group_name,
            error_y=dict(type='data', array=std_devs, visible=True),  # Error bars (Standard Deviation)
            line=dict(width=2),
            marker=dict(size=6)
        ))

# Create layout for the graphs with increased width and tick angle of 90 degrees
layout = go.Layout(
    title="Pooled Averages of Parameters for Different Group Combinations",
    xaxis=dict(
        title='Time (minutes)',
        tickvals=time_points_to_keep,  # Set ticks at every point in the list
        ticktext=[str(time) for time in time_points_to_keep],  # Label each tick with the time point
        tickangle=90  # Rotate x-axis ticks by 90 degrees
    ),
    yaxis=dict(title='Value'),
    showlegend=True,
    width=900  # Set the width to 900
)

# Create the figure for Total Vasopressin
fig_vasopressin = go.Figure(data=traces_vasopressin, layout=layout)
fig_vasopressin.update_layout(title='Vasopressin')

# Create the figure for Plasmalyte
fig_plasmalyte = go.Figure(data=traces_plasmalyte, layout=layout)
fig_plasmalyte.update_layout(title='Plasmalyte')

# Create the figure for Norepi
fig_norepi = go.Figure(data=traces_norepi, layout=layout)
fig_norepi.update_layout(title='Norepinephrine')

# Save the figures as HTML files
fig_vasopressin.write_html(os.path.join(output_dir, 'vasopressin.html'))
fig_plasmalyte.write_html(os.path.join(output_dir, 'plasmalyte.html'))
fig_norepi.write_html(os.path.join(output_dir, 'norepi.html'))

# Show the figures (optional)
fig_vasopressin.show()
fig_plasmalyte.show()
fig_norepi.show()


In [ ]:
import os
import pandas as pd
import plotly.graph_objects as go
import numpy as np

# Create the "drug concentrations" directory if it doesn't exist
output_dir = "drug concentrations"
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

# Read the categorization data from 'Experiment Tracker.xlsx'
df = pd.read_excel('Experiment Tracker.xlsx', sheet_name=['T30 categorization', 'T60 categorization'])

# Categorization at time=30
group_10 = set(df['T30 categorization']['Group_10'].dropna())
group_20 = set(df['T30 categorization']['Group_20'].dropna())
group_30 = set(df['T30 categorization']['Group_30'].dropna())

# Categorization at time=60
full_occlusion = set(df['T60 categorization']['Full_Occlusion'].dropna())
partial_occlusion = set(df['T60 categorization']['Partial_Occlusion'].dropna())
no_occlusion = set(df['T60 categorization']['No_Occlusion'].dropna())

# Find overlapping subjects for each combination at time=60
hemorrhage_10_full_occlusion = group_10.intersection(full_occlusion)
hemorrhage_10_partial_occlusion = group_10.intersection(partial_occlusion)
hemorrhage_10_no_occlusion = group_10.intersection(no_occlusion)

hemorrhage_20_full_occlusion = group_20.intersection(full_occlusion)
hemorrhage_20_partial_occlusion = group_20.intersection(partial_occlusion)
hemorrhage_20_no_occlusion = group_20.intersection(no_occlusion)

hemorrhage_30_full_occlusion = group_30.intersection(full_occlusion)
hemorrhage_30_partial_occlusion = group_30.intersection(partial_occlusion)
hemorrhage_30_no_occlusion = group_30.intersection(no_occlusion)

# Now load the Excel file containing the main data (processed data)
file_path = "Processed_Data_with_Vasopressin.xlsx"
xls = pd.ExcelFile(file_path)

# Filter sheets that start with "ERNE" and not "ERDE"
erne_sheets = [sheet for sheet in xls.sheet_names if sheet.startswith("ERNE")]

# Dictionary to store dynamically generated variables
data_dict = {}

# Loop through the filtered sheets and store them in the dictionary
for sheet in erne_sheets:
    new_sheet_name = sheet.replace("-", "")
    df = xls.parse(sheet)
    required_columns = [
        "Time", "Cumulative Vasopressin", "Plasmalyte", "Norepi",
        "Prox_Sys_PowerLab_PC", "Prox_Dia_PowerLab_PC", "Prox_Mean_PowerLab_PC",
        "Dist_Sys_PowerLab_PC", "Dist_Dia_PowerLab_PC", "Dist_Mean_PowerLab_PC",
        "Dist_Sys_LPF_PowerLab_PC", "Dist_Dia_LPF_PowerLab_PC", "Dist_Mean_LPF_PowerLab_PC"
    ]
    df = df[required_columns]
    # Remove rows where 'Time' is greater than 240
    df = df[df["Time"] <= 240]
    data_dict[new_sheet_name] = df

# Define a function to manually calculate mean and standard deviation for each group combination, keeping only specified time points
def calculate_mean_and_std(group_data, time_col='Time', value_col='Cumulative Vasopressin', time_points=None):
    """
    Calculate the mean and standard deviation for a given group, only for specified time points.
    """
    group_values = []
    times = []
    
    # Aggregating data for each time point
    for group in group_data:
        if group in data_dict:
            df = data_dict[group]
            group_times = df[time_col].unique()  # Time points
            # Filter to keep only the specified time points, if any
            if time_points:
                group_times = [time for time in group_times if time in time_points]
            for time in group_times:
                group_at_time = df[df[time_col] == time]
                group_values.extend(group_at_time[value_col].tolist())
                times.extend([time] * len(group_at_time[value_col]))

    # Calculate mean and standard deviation for each time point
    unique_times = sorted(set(times))
    means = []
    std_devs = []
    
    for time in unique_times:
        values_at_time = [group_values[i] for i, t in enumerate(times) if t == time]
        means.append(np.mean(values_at_time))
        std_devs.append(np.std(values_at_time))  # Standard deviation

    return unique_times, means, std_devs

# Define your list of time points to keep (as you provided)
time_points_to_keep = [0,5,10,15,20,25,30,60,65,75,85,120,180,240]

# Create combined groups (time=60 categorization)
group_combinations = {
    "10% Hemorrhage + Full Occlusion": hemorrhage_10_full_occlusion,
    "10% Hemorrhage + Partial Occlusion": hemorrhage_10_partial_occlusion,
    "10% Hemorrhage + No Occlusion": hemorrhage_10_no_occlusion,
    "20% Hemorrhage + Full Occlusion": hemorrhage_20_full_occlusion,
    "20% Hemorrhage + Partial Occlusion": hemorrhage_20_partial_occlusion,
    "20% Hemorrhage + No Occlusion": hemorrhage_20_no_occlusion,
    "30% Hemorrhage + Full Occlusion": hemorrhage_30_full_occlusion,
    "30% Hemorrhage + Partial Occlusion": hemorrhage_30_partial_occlusion,
    "30% Hemorrhage + No Occlusion": hemorrhage_30_no_occlusion,
}

# List of columns to graph
columns_to_graph = [
    "Cumulative Vasopressin", "Plasmalyte", "Norepi",
    "Prox_Sys_PowerLab_PC", "Prox_Dia_PowerLab_PC", "Prox_Mean_PowerLab_PC",
    "Dist_Sys_PowerLab_PC", "Dist_Dia_PowerLab_PC", "Dist_Mean_PowerLab_PC",
    "Dist_Sys_LPF_PowerLab_PC", "Dist_Dia_LPF_PowerLab_PC", "Dist_Mean_LPF_PowerLab_PC"
]

# Prepare the figures and save them as HTML files for all columns
for column in columns_to_graph:
    traces = []
    for group_name, group_data in group_combinations.items():
        if group_data:  # Only calculate if there is data
            times, means, std_devs = calculate_mean_and_std(group_data, value_col=column, time_points=time_points_to_keep)
            
            # Add to the plot
            traces.append(go.Scatter(
                x=times, y=means,
                mode='lines+markers',
                name=group_name,
                error_y=dict(type='data', array=std_devs, visible=True),  # Error bars (Standard Deviation)
                line=dict(width=2),
                marker=dict(size=6)
            ))

    # Create layout for the graph with increased width and tick angle of 90 degrees
    layout = go.Layout(
        title=f"{column} vs Time",
        xaxis=dict(
            title='Time (minutes)',
            tickvals=time_points_to_keep,  # Set ticks at every point in the list
            ticktext=[str(time) for time in time_points_to_keep],  # Label each tick with the time point
            tickangle=90  # Rotate x-axis ticks by 90 degrees
        ),
        yaxis=dict(title=column),  # Use column name as y-axis label
        showlegend=True,
        width=900  # Set the width to 900
    )

    # Create the figure
    fig = go.Figure(data=traces, layout=layout)

    # Save the figure as an HTML file
    fig.write_html(os.path.join(output_dir, f"{column.replace(' ', '_')}.html"))

    # Show the figure (optional)
    fig.show()


# Fresh AF style

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Output directory for PNGs
output_dir = "drug concentrations"
os.makedirs(output_dir, exist_ok=True)

# Read categorization data
df = pd.read_excel('Experiment Tracker.xlsx', sheet_name=['T30 categorization', 'T60 categorization'])

# Categorization at time=30
group_10 = set(df['T30 categorization']['Group_10'].dropna())
group_20 = set(df['T30 categorization']['Group_20'].dropna())
group_30 = set(df['T30 categorization']['Group_30'].dropna())

# Categorization at time=60
full_occlusion = set(df['T60 categorization']['Full_Occlusion'].dropna())
partial_occlusion = set(df['T60 categorization']['Partial_Occlusion'].dropna())
no_occlusion = set(df['T60 categorization']['No_Occlusion'].dropna())

# Mapping groups to hemorrhage levels
group_combinations = {
    "10": {
        "Full Occlusion": group_10.intersection(full_occlusion),
        "Partial Occlusion": group_10.intersection(partial_occlusion),
        "No Occlusion": group_10.intersection(no_occlusion)
    },
    "20": {
        "Full Occlusion": group_20.intersection(full_occlusion),
        "Partial Occlusion": group_20.intersection(partial_occlusion),
        "No Occlusion": group_20.intersection(no_occlusion)
    },
    "30": {
        "Full Occlusion": group_30.intersection(full_occlusion),
        "Partial Occlusion": group_30.intersection(partial_occlusion),
        "No Occlusion": group_30.intersection(no_occlusion)
    }
}

# Load processed data
xls = pd.ExcelFile("Processed_Data_with_Vasopressin.xlsx")
erne_sheets = [sheet for sheet in xls.sheet_names if sheet.startswith("ERNE")]
data_dict = {}

for sheet in erne_sheets:
    df_sheet = xls.parse(sheet)
    new_sheet_name = sheet.replace("-", "")
    required_columns = [
        "Time", "Cumulative Vasopressin", "Plasmalyte", "Norepi",
        "Prox_Sys_PowerLab_PC", "Prox_Dia_PowerLab_PC", "Prox_Mean_PowerLab_PC",
        "Dist_Sys_PowerLab_PC", "Dist_Dia_PowerLab_PC", "Dist_Mean_PowerLab_PC",
        "Dist_Sys_LPF_PowerLab_PC", "Dist_Dia_LPF_PowerLab_PC", "Dist_Mean_LPF_PowerLab_PC"
    ]
    df_sheet = df_sheet[required_columns]
    df_sheet = df_sheet[df_sheet["Time"] <= 240]
    data_dict[new_sheet_name] = df_sheet

# Calculation function
def calculate_mean_and_std(group_data, time_col='Time', value_col='Cumulative Vasopressin', time_points=None):
    group_values = {}
    
    for subject in group_data:
        if subject in data_dict:
            df = data_dict[subject]
            df_filtered = df[df[time_col].isin(time_points)]
            for _, row in df_filtered.iterrows():
                time = row[time_col]
                val = row[value_col]
                group_values.setdefault(time, []).append(val)

    times = sorted(group_values.keys())
    means = [np.mean(group_values[t]) for t in times]
    stds = [np.std(group_values[t]) for t in times]
    
    return times, means, stds

# Settings
time_points_to_keep = [0,5,10,15,20,25,30,60,65,75,85,120,180,240]
columns_to_graph = [
    "Cumulative Vasopressin", "Plasmalyte", "Norepi",
    "Prox_Sys_PowerLab_PC", "Prox_Dia_PowerLab_PC", "Prox_Mean_PowerLab_PC",
    "Dist_Sys_PowerLab_PC", "Dist_Dia_PowerLab_PC", "Dist_Mean_PowerLab_PC",
    "Dist_Sys_LPF_PowerLab_PC", "Dist_Dia_LPF_PowerLab_PC", "Dist_Mean_LPF_PowerLab_PC"
]

# Generate plots by hemorrhage level
for hemorrhage_level in ["10", "20", "30"]:
    hemorrhage_dir = os.path.join(output_dir, f"{hemorrhage_level}_percent_hemorrhage")
    os.makedirs(hemorrhage_dir, exist_ok=True)

    for column in columns_to_graph:
        plt.figure(figsize=(12, 6))
        for occlusion_type, group_data in group_combinations[hemorrhage_level].items():
            if group_data:
                times, means, stds = calculate_mean_and_std(group_data, value_col=column, time_points=time_points_to_keep)
                plt.errorbar(times, means, yerr=stds, label=occlusion_type, marker='o', capsize=3)

        plt.title(f"{column} vs Time ({hemorrhage_level}% Hemorrhage)")
        plt.xlabel("Time (minutes)")
        plt.ylabel(column)
        plt.xticks(time_points_to_keep, rotation=90)
        plt.legend()
        plt.tight_layout()

        # Save PNG
        file_name = f"{column.replace(' ', '_')}.png"
        plt.savefig(os.path.join(hemorrhage_dir, file_name), dpi=300)
        plt.close()


# Old School Style

In [ ]:
import os
import pandas as pd
import numpy as np
import plotly.graph_objects as go

# Create output directory
output_folder = "Individual Hemorrhage Level Vasopressin Analysis"
os.makedirs(output_folder, exist_ok=True)

# Load categorization data
cat_data = pd.read_excel('Experiment Tracker.xlsx', sheet_name=['T30 categorization', 'T60 categorization'])

# Define hemorrhage groups
group_10 = set(cat_data['T30 categorization']['Group_10'].dropna())
group_20 = set(cat_data['T30 categorization']['Group_20'].dropna())
group_30 = set(cat_data['T30 categorization']['Group_30'].dropna())

# Define occlusion categories
full_occlusion = set(cat_data['T60 categorization']['Full_Occlusion'].dropna())
partial_occlusion = set(cat_data['T60 categorization']['Partial_Occlusion'].dropna())
no_occlusion = set(cat_data['T60 categorization']['No_Occlusion'].dropna())

# Map subjects by hemorrhage level and occlusion type
group_combinations = {
    "10": {
        "Full": group_10 & full_occlusion,
        "Partial": group_10 & partial_occlusion,
        "None": group_10 & no_occlusion
    },
    "20": {
        "Full": group_20 & full_occlusion,
        "Partial": group_20 & partial_occlusion,
        "None": group_20 & no_occlusion
    },
    "30": {
        "Full": group_30 & full_occlusion,
        "Partial": group_30 & partial_occlusion,
        "None": group_30 & no_occlusion
    }
}

# Load main data
xls = pd.ExcelFile("Processed_Data_with_Vasopressin.xlsx")
sheet_names = [s for s in xls.sheet_names if s.startswith("ERNE")]
data_dict = {}

for sheet in sheet_names:
    clean_name = sheet.replace("-", "")
    df = xls.parse(sheet)
    required_cols = [
        "Time", "Cumulative Vasopressin", "Plasmalyte", "Norepi",
        "Prox_Sys_PowerLab_PC", "Prox_Dia_PowerLab_PC", "Prox_Mean_PowerLab_PC",
        "Dist_Sys_PowerLab_PC", "Dist_Dia_PowerLab_PC", "Dist_Mean_PowerLab_PC",
        "Dist_Sys_LPF_PowerLab_PC", "Dist_Dia_LPF_PowerLab_PC", "Dist_Mean_LPF_PowerLab_PC"
    ]
    df = df[required_cols]
    df = df[df["Time"] <= 240]
    data_dict[clean_name] = df

# Time points to include
time_points = [0,5,10,15,20,25,30,60,65,75,85,120,180,240]

# Define color and symbol mappings
occlusion_colors = {'None': 'black', 'Partial': 'blue', 'Full': 'red'}
occlusion_symbols = {'None': 'circle', 'Partial': 'triangle-up', 'Full': 'square'}

# Columns to plot
columns_to_plot = [
    "Cumulative Vasopressin", "Plasmalyte", "Norepi",
    "Prox_Sys_PowerLab_PC", "Prox_Dia_PowerLab_PC", "Prox_Mean_PowerLab_PC",
    "Dist_Sys_PowerLab_PC", "Dist_Dia_PowerLab_PC", "Dist_Mean_PowerLab_PC",
    "Dist_Sys_LPF_PowerLab_PC", "Dist_Dia_LPF_PowerLab_PC", "Dist_Mean_LPF_PowerLab_PC"
]

# Function to calculate means and stds
def calculate_means(group, column):
    time_series = {t: [] for t in time_points}
    
    for subject in group:
        if subject in data_dict:
            df = data_dict[subject]
            for t in time_points:
                values = df[df['Time'] == t][column]
                if not values.empty:
                    time_series[t].append(values.values[0])
    
    times = []
    means = []
    stds = []
    
    for t in time_points:
        values = time_series[t]
        if values:
            times.append(t)
            means.append(np.mean(values))
            stds.append(np.std(values))
    
    return times, means, stds

# Plotting loop
for hemorrhage_level in ["10", "20", "30"]:
    for column in columns_to_plot:
        fig = go.Figure()
        
        for occlusion_type in ["Partial", "Full", "None"]:
            group = group_combinations[hemorrhage_level][occlusion_type]
            if not group:
                continue

            times, means, stds = calculate_means(group, column)
            display_name = "No Occlusion" if occlusion_type == "None" else f"{occlusion_type} Occlusion"

            fig.add_trace(go.Scatter(
                x=times,
                y=means,
                mode="lines+markers",
                name=display_name,
                error_y=dict(type="data", array=stds, visible=True),
                marker=dict(color=occlusion_colors[occlusion_type], symbol=occlusion_symbols[occlusion_type], size=8),
                line=dict(width=2)
            ))

        fig.update_layout(
            title=f"{hemorrhage_level}% Hemorrhage {column}",
            title_font=dict(size=24),
            xaxis_title="Time (min)",
            yaxis_title=column,
            legend_title="Occlusion Type",
            xaxis=dict(
                tickmode='array',
                tickvals=time_points,
                tickangle=90,
                tickfont=dict(size=16),
                title_font=dict(size=24)
            ),
            yaxis=dict(
                tickfont=dict(size=16),
                title_font=dict(size=24),
                range=[0, 200]
#                 dtick=0.25
                
            ),
            width=1200,
            height=600
        )

        # Save plot
        file_base = f"{column.replace(' ', '_')}_Hemorrhage_{hemorrhage_level}"
        fig.write_image(os.path.join(output_folder, f"{file_base}.png"))
        fig.write_html(os.path.join(output_folder, f"{file_base}.html"))

print('Done')

# Occlusion only remake

In [ ]:
import os
import pandas as pd
import numpy as np
import plotly.graph_objects as go

# Create output directory
output_folder = "Individual Occlusion Level Vasopressin Analysis"
os.makedirs(output_folder, exist_ok=True)

# Load categorization data
cat_data = pd.read_excel('Experiment Tracker.xlsx', sheet_name=['T30 categorization', 'T60 categorization'])

# Define hemorrhage groups
group_10 = set(cat_data['T30 categorization']['Group_10'].dropna())
group_20 = set(cat_data['T30 categorization']['Group_20'].dropna())
group_30 = set(cat_data['T30 categorization']['Group_30'].dropna())

# Define occlusion categories
full_occlusion = set(cat_data['T60 categorization']['Full_Occlusion'].dropna())
partial_occlusion = set(cat_data['T60 categorization']['Partial_Occlusion'].dropna())
no_occlusion = set(cat_data['T60 categorization']['No_Occlusion'].dropna())

# Map subjects by occlusion type and hemorrhage level
occlusion_combinations = {
    "Full": {
        "10": group_10 & full_occlusion,
        "20": group_20 & full_occlusion,
        "30": group_30 & full_occlusion
    },
    "Partial": {
        "10": group_10 & partial_occlusion,
        "20": group_20 & partial_occlusion,
        "30": group_30 & partial_occlusion
    },
    "None": {
        "10": group_10 & no_occlusion,
        "20": group_20 & no_occlusion,
        "30": group_30 & no_occlusion
    }
}

# Load main data
xls = pd.ExcelFile("Processed_Data_with_Vasopressin.xlsx")
sheet_names = [s for s in xls.sheet_names if s.startswith("ERNE")]
data_dict = {}

for sheet in sheet_names:
    clean_name = sheet.replace("-", "")
    df = xls.parse(sheet)
    required_cols = [
        "Time", "Cumulative Vasopressin", "Plasmalyte", "Norepi",
        "Prox_Sys_PowerLab_PC", "Prox_Dia_PowerLab_PC", "Prox_Mean_PowerLab_PC",
        "Dist_Sys_PowerLab_PC", "Dist_Dia_PowerLab_PC", "Dist_Mean_PowerLab_PC",
        "Dist_Sys_LPF_PowerLab_PC", "Dist_Dia_LPF_PowerLab_PC", "Dist_Mean_LPF_PowerLab_PC"
    ]
    df = df[required_cols]
    df = df[df["Time"] <= 240]
    data_dict[clean_name] = df

# Time points to include
time_points = [0,5,10,15,20,25,30,60,65,75,85,120,180,240]

# Define colors and markers for hemorrhage levels
hemorrhage_colors = {'10': 'brown', '20': 'orange', '30': 'purple'}
hemorrhage_symbols = {'10': 'circle', '20': 'square', '30': 'diamond'}

# Columns to plot
columns_to_plot = [
    "Cumulative Vasopressin", "Plasmalyte", "Norepi",
    "Prox_Sys_PowerLab_PC", "Prox_Dia_PowerLab_PC", "Prox_Mean_PowerLab_PC",
    "Dist_Sys_PowerLab_PC", "Dist_Dia_PowerLab_PC", "Dist_Mean_PowerLab_PC",
    "Dist_Sys_LPF_PowerLab_PC", "Dist_Dia_LPF_PowerLab_PC", "Dist_Mean_LPF_PowerLab_PC"
]

# Function to calculate means and stds
def calculate_means(group, column):
    time_series = {t: [] for t in time_points}
    
    for subject in group:
        if subject in data_dict:
            df = data_dict[subject]
            for t in time_points:
                values = df[df['Time'] == t][column]
                if not values.empty:
                    time_series[t].append(values.values[0])
    
    times = []
    means = []
    stds = []
    
    for t in time_points:
        values = time_series[t]
        if values:
            times.append(t)
            means.append(np.mean(values))
            stds.append(np.std(values))
    
    return times, means, stds

# Plotting loop: one figure per occlusion type
for occlusion_type in ["Full", "Partial", "None"]:
    for column in columns_to_plot:
        fig = go.Figure()
        
        for hemorrhage_level in ["10", "20", "30"]:
            group = occlusion_combinations[occlusion_type][hemorrhage_level]
            if not group:
                continue

            times, means, stds = calculate_means(group, column)
            label = f"{hemorrhage_level}% Hemorrhage"

            fig.add_trace(go.Scatter(
                x=times,
                y=means,
                mode="lines+markers",
                name=label,
                error_y=dict(type="data", array=stds, visible=True),
                marker=dict(color=hemorrhage_colors[hemorrhage_level], symbol=hemorrhage_symbols[hemorrhage_level], size=8),
                line=dict(width=2)
            ))

        # Legend fix
        occlusion_label = "No Occlusion" if occlusion_type == "None" else f"{occlusion_type} Occlusion"

        fig.update_layout(
            title=f"{column} ({occlusion_label})",
            title_font=dict(size=24),
            xaxis_title="Time (min)",
            yaxis_title=column,
            legend_title="Hemorrhage Level",
            xaxis=dict(
                tickmode='array',
                tickvals=time_points,
                tickangle=90,
                tickfont=dict(size=16),
                title_font=dict(size=24)
            ),
            yaxis=dict(
                tickfont=dict(size=16),
                title_font=dict(size=24),
                range=[0, 200]
            ),
            width=1200,
            height=600
        )

        # Save files
        file_base = f"{column.replace(' ', '_')}_Occlusion_{occlusion_type}"
        fig.write_image(os.path.join(output_folder, f"{file_base}.png"))
        fig.write_html(os.path.join(output_folder, f"{file_base}.html"))
print('Done')

# Other 

In [ ]:
import os
import pandas as pd
import numpy as np

# Create the "drug concentrations" directory if it doesn't exist
output_dir = "drug concentrations"
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

# Read the categorization data from 'Experiment Tracker.xlsx'
df = pd.read_excel('Experiment Tracker.xlsx', sheet_name=['T30 categorization', 'T60 categorization'])

# Categorization at time=30
group_10 = set(df['T30 categorization']['Group_10'].dropna())
group_20 = set(df['T30 categorization']['Group_20'].dropna())
group_30 = set(df['T30 categorization']['Group_30'].dropna())

# Categorization at time=60
full_occlusion = set(df['T60 categorization']['Full_Occlusion'].dropna())
partial_occlusion = set(df['T60 categorization']['Partial_Occlusion'].dropna())
no_occlusion = set(df['T60 categorization']['No_Occlusion'].dropna())

# Load the processed data file
file_path = "Processed_Data_with_Vasopressin.xlsx"
xls = pd.ExcelFile(file_path)

# Filter sheets that start with "ERNE" and not "ERDE"
erne_sheets = [sheet for sheet in xls.sheet_names if sheet.startswith("ERNE")]

# List to store data before concatenation
data_list = []

# Loop through the filtered sheets and store them in the list
for sheet in erne_sheets:
    df = xls.parse(sheet)
    required_columns = [
        "Time", "Cumulative Vasopressin", "Plasmalyte", "Norepi",
        "Prox_Sys_PowerLab_PC", "Prox_Dia_PowerLab_PC", "Prox_Mean_PowerLab_PC",
        "Dist_Sys_PowerLab_PC", "Dist_Dia_PowerLab_PC", "Dist_Mean_PowerLab_PC",
        "Dist_Sys_LPF_PowerLab_PC", "Dist_Dia_LPF_PowerLab_PC", "Dist_Mean_LPF_PowerLab_PC"
    ]
    
    df = df[required_columns]
    
    # Remove rows where 'Time' is greater than 240
    df = df[df["Time"] <= 240]
    
    # Add a column to track the source sheet name
    df["Source"] = sheet
    
    # Append to the list
    data_list.append(df)

# Concatenate all DataFrames into one
final_df = pd.concat(data_list, ignore_index=True)

# Load Urine Output Data
urine_file = "Urine Output.xlsx"
urine_df = pd.read_excel(urine_file, dtype=str)  # Read all values as strings to preserve text

# Initialize the "Urine Output" column with blank values
final_df["Urine Output"] = "N/A"

# Iterate through the urine output file to map values to the final_df
for _, row in urine_df.iterrows():
    time_point = row["Time"]  # Get the time value from urine data
    for col in urine_df.columns[1:]:  # Skip the "Time" column
        subject_id = col  # Column names in urine file match "Source" in final_df
        mask = (final_df["Time"] == float(time_point)) & (final_df["Source"] == subject_id)
        final_df.loc[mask, "Urine Output"] = row[col]  # Assign urine output value as string

# Save the final DataFrame to Excel
output_file = "Processed_Data_Output.xlsx"
with pd.ExcelWriter(output_file, engine="xlsxwriter") as writer:
    final_df.to_excel(writer, sheet_name="All Data", index=False)

print(f"Data successfully saved in '{output_file}' under sheet 'All Data' with Urine Output column.")


In [3]:
import os
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ----------------------------------------
# Output directory
# ----------------------------------------
output_dir = "drug_concentrations_panels"
os.makedirs(output_dir, exist_ok=True)

# ----------------------------------------
# Load categorization data
# ----------------------------------------
df = pd.read_excel('Experiment Tracker.xlsx', sheet_name=['T30 categorization', 'T60 categorization'])

# Time=30 groups
group_10 = set(df['T30 categorization']['Group_10'].dropna())
group_20 = set(df['T30 categorization']['Group_20'].dropna())
group_30 = set(df['T30 categorization']['Group_30'].dropna())

# Time=60 occlusions
full_occlusion = set(df['T60 categorization']['Full_Occlusion'].dropna())
partial_occlusion = set(df['T60 categorization']['Partial_Occlusion'].dropna())
no_occlusion = set(df['T60 categorization']['No_Occlusion'].dropna())

# Mapping hemorrhage level × occlusion group → subject IDs
group_combinations = {
    "10": {
        "Full": group_10.intersection(full_occlusion),
        "Partial": group_10.intersection(partial_occlusion),
        "No": group_10.intersection(no_occlusion)
    },
    "20": {
        "Full": group_20.intersection(full_occlusion),
        "Partial": group_20.intersection(partial_occlusion),
        "No": group_20.intersection(no_occlusion)
    },
    "30": {
        "Full": group_30.intersection(full_occlusion),
        "Partial": group_30.intersection(partial_occlusion),
        "No": group_30.intersection(no_occlusion)
    }
}

# ----------------------------------------
# Load processed ERNE data
# ----------------------------------------
xls = pd.ExcelFile("Processed_Data_with_Vasopressin.xlsx")
erne_sheets = [sheet for sheet in xls.sheet_names if sheet.startswith("ERNE")]
data_dict = {}

for sheet in erne_sheets:
    df_sheet = xls.parse(sheet)
    new_sheet_name = sheet.replace("-", "")
    required_columns = [
        "Time", "Cumulative Vasopressin", "Plasmalyte", "Norepi",
        # "Prox_Sys_PowerLab_PC", "Prox_Dia_PowerLab_PC", "Prox_Mean_PowerLab_PC",
        # "Dist_Sys_PowerLab_PC", "Dist_Dia_PowerLab_PC", "Dist_Mean_PowerLab_PC",
        # "Dist_Sys_LPF_PowerLab_PC", "Dist_Dia_LPF_PowerLab_PC", "Dist_Mean_LPF_PowerLab_PC"
    ]
    df_sheet = df_sheet[required_columns]
    df_sheet = df_sheet[df_sheet["Time"] <= 240]
    data_dict[new_sheet_name] = df_sheet

# ----------------------------------------
# Helper function for mean and std computation
# ----------------------------------------
def calculate_mean_and_std(group_data, time_col='Time', value_col='Cumulative Vasopressin', time_points=None):
    group_values = {}
    for subject in group_data:
        if subject in data_dict:
            df = data_dict[subject]
            df_filtered = df[df[time_col].isin(time_points)]
            for _, row in df_filtered.iterrows():
                time = row[time_col]
                val = row[value_col]
                group_values.setdefault(time, []).append(val)

    if not group_values:
        return [], [], []

    times = sorted(group_values.keys())
    means = [np.mean(group_values[t]) for t in times]
    stds = [np.std(group_values[t]) for t in times]
    return times, means, stds

# ----------------------------------------
# Plotting parameters
# ----------------------------------------
time_points_to_keep = [0,5,10,15,20,25,30,60,65,75,85,120,180,240]
columns_to_graph = [
    "Cumulative Vasopressin", "Plasmalyte", "Norepi",
    # "Prox_Sys_PowerLab_PC", "Prox_Dia_PowerLab_PC", "Prox_Mean_PowerLab_PC",
    # "Dist_Sys_PowerLab_PC", "Dist_Dia_PowerLab_PC", "Dist_Mean_PowerLab_PC",
    # "Dist_Sys_LPF_PowerLab_PC", "Dist_Dia_LPF_PowerLab_PC", "Dist_Mean_LPF_PowerLab_PC"
]

# Consistent color/marker scheme
marker_info = {
    "10": {"color": "orange", "marker": "o"},
    "20": {"color": "purple", "marker": "s"},
    "30": {"color": "brown", "marker": "D"},
}
occlusion_order = ["No", "Partial", "Full"]

# ----------------------------------------
# Multi-panel plotting function
# ----------------------------------------
def create_panel_plots():
    for column in columns_to_graph:
        fig, axes = plt.subplots(1, len(occlusion_order), figsize=(18, 6), sharey=True)
        handles, labels = [], []

        for ax, occlusion_type in zip(axes, occlusion_order):
            for hemorrhage_level in ["10", "20", "30"]:
                group_data = group_combinations[hemorrhage_level].get(occlusion_type, set())
                if not group_data:
                    continue

                times, means, stds = calculate_mean_and_std(group_data, value_col=column, time_points=time_points_to_keep)
                if not times:
                    continue

                color = marker_info[hemorrhage_level]["color"]
                marker = marker_info[hemorrhage_level]["marker"]
                label_name = f"{hemorrhage_level}% Hemorrhage"

                # Plot mean line
                line, = ax.plot(
                    times, means, color=color, marker=marker, linewidth=2,
                    markersize=8, label=label_name
                )

                # Shaded ±1 SD band
                ax.fill_between(times, np.array(means)-np.array(stds), np.array(means)+np.array(stds),
                                color=color, alpha=0.2)

                if label_name not in labels:
                    handles.append(line)
                    labels.append(label_name)

            ax.set_title(f"{occlusion_type} Occlusion", fontsize=18)
            ax.set_xlabel("Time (min)", fontsize=16)
            ax.grid(True, linestyle='--', linewidth=0.5)
            ax.tick_params(axis='both', labelsize=14)

        axes[0].set_ylabel(column, fontsize=16)
        axes[-1].legend(handles, labels, loc='center left', bbox_to_anchor=(1.02, 0.5), fontsize=12)

        plt.suptitle(f"{column} vs Time (All Hemorrhage Levels)", fontsize=20)
        plt.tight_layout(rect=[0, 0, 0.88, 0.95])

        output_path = os.path.join(output_dir, f"{column.replace(' ', '_')}_panel.png")
        plt.savefig(output_path, dpi=300, bbox_inches='tight')
        plt.close()

# ----------------------------------------
# Run plotting
# ----------------------------------------
create_panel_plots()


In [5]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ----------------------------------------
# Output directory
# ----------------------------------------
output_dir = "drug_concentrations_by_hemorrhage"
os.makedirs(output_dir, exist_ok=True)

# ----------------------------------------
# Load categorization data
# ----------------------------------------
df = pd.read_excel('Experiment Tracker.xlsx', sheet_name=['T30 categorization', 'T60 categorization'])

# Time=30 groups
group_10 = set(df['T30 categorization']['Group_10'].dropna())
group_20 = set(df['T30 categorization']['Group_20'].dropna())
group_30 = set(df['T30 categorization']['Group_30'].dropna())

# Time=60 occlusion group membership
full_occlusion = set(df['T60 categorization']['Full_Occlusion'].dropna())
partial_occlusion = set(df['T60 categorization']['Partial_Occlusion'].dropna())
no_occlusion = set(df['T60 categorization']['No_Occlusion'].dropna())

# Mapping hemorrhage × occlusion → subject IDs
group_combinations = {
    "10": {
        "Full": group_10.intersection(full_occlusion),
        "Partial": group_10.intersection(partial_occlusion),
        "No": group_10.intersection(no_occlusion)
    },
    "20": {
        "Full": group_20.intersection(full_occlusion),
        "Partial": group_20.intersection(partial_occlusion),
        "No": group_20.intersection(no_occlusion)
    },
    "30": {
        "Full": group_30.intersection(full_occlusion),
        "Partial": group_30.intersection(partial_occlusion),
        "No": group_30.intersection(no_occlusion)
    }
}

# ----------------------------------------
# Load processed ERNE data
# ----------------------------------------
xls = pd.ExcelFile("Processed_Data_with_Vasopressin.xlsx")
erne_sheets = [s for s in xls.sheet_names if s.startswith("ERNE")]
data_dict = {}

for sheet in erne_sheets:
    df_sheet = xls.parse(sheet)
    new_name = sheet.replace("-", "")
    required_cols = [
        "Time", "Cumulative Vasopressin", "Plasmalyte", "Norepi",
        "Prox_Sys_PowerLab_PC", "Prox_Dia_PowerLab_PC", "Prox_Mean_PowerLab_PC",
        "Dist_Sys_PowerLab_PC", "Dist_Dia_PowerLab_PC", "Dist_Mean_PowerLab_PC",
        "Dist_Sys_LPF_PowerLab_PC", "Dist_Dia_LPF_PowerLab_PC", "Dist_Mean_LPF_PowerLab_PC"
    ]
    df_sheet = df_sheet[required_cols]
    df_sheet = df_sheet[df_sheet["Time"] <= 240]
    data_dict[new_name] = df_sheet

# ----------------------------------------
# Compute mean and std at each timepoint
# ----------------------------------------
def calculate_mean_and_std(group_data, time_col='Time', value_col='Cumulative Vasopressin', time_points=None):
    group_values = {}
    for subject in group_data:
        if subject in data_dict:
            df = data_dict[subject]
            df_filt = df[df[time_col].isin(time_points)]
            for _, row in df_filt.iterrows():
                t, val = row[time_col], row[value_col]
                group_values.setdefault(t, []).append(val)

    if not group_values:
        return [], [], []

    times = sorted(group_values.keys())
    means = [np.mean(group_values[t]) for t in times]
    stds = [np.std(group_values[t]) for t in times]
    return times, means, stds

# ----------------------------------------
# Plot settings
# ----------------------------------------
time_points_to_keep = [0,5,10,15,20,25,30,60,65,75,85,120,180,240]
columns_to_graph = [
    "Cumulative Vasopressin", "Plasmalyte", "Norepi",

]

occlusion_order = ["No", "Partial", "Full"]
occlusion_colors = {"No": "steelblue", "Partial": "darkorange", "Full": "darkred"}
occlusion_markers = {"No": "o", "Partial": "s", "Full": "D"}

# ----------------------------------------
# Multi-panel plotting by hemorrhage level
# ----------------------------------------
def create_panel_plots_by_hemorrhage():
    for hemorrhage_level in ["10", "20", "30"]:
        hemorrhage_dir = os.path.join(output_dir, f"{hemorrhage_level}_percent_hemorrhage")
        os.makedirs(hemorrhage_dir, exist_ok=True)

        for column in columns_to_graph:
            fig, axes = plt.subplots(1, len(occlusion_order), figsize=(18, 6), sharey=True)
            handles, labels = [], []

            for ax, occ_type in zip(axes, occlusion_order):
                group_data = group_combinations[hemorrhage_level].get(occ_type, set())
                if not group_data:
                    ax.set_visible(False)
                    continue

                times, means, stds = calculate_mean_and_std(
                    group_data, value_col=column, time_points=time_points_to_keep
                )
                if not times:
                    ax.set_visible(False)
                    continue

                color = occlusion_colors[occ_type]
                marker = occlusion_markers[occ_type]

                line, = ax.plot(
                    times, means, color=color, marker=marker, linewidth=2, markersize=8,
                    label=f"{occ_type} Occlusion"
                )
                ax.fill_between(times, np.array(means) - np.array(stds),
                                np.array(means) + np.array(stds),
                                color=color, alpha=0.2)

                ax.set_title(f"{occ_type} Occlusion", fontsize=18)
                ax.set_xlabel("Time (min)", fontsize=16)
                ax.grid(True, linestyle='--', linewidth=0.5)
                ax.tick_params(axis='both', labelsize=14)

                handles.append(line)
                labels.append(f"{occ_type} Occlusion")

            axes[0].set_ylabel(column, fontsize=16)
            axes[-1].legend(handles, labels, loc='center left', bbox_to_anchor=(1.02, 0.5), fontsize=12)
            plt.suptitle(f"{column} vs Time ({hemorrhage_level}% Hemorrhage)", fontsize=20)
            plt.tight_layout(rect=[0, 0, 0.88, 0.95])

            output_file = os.path.join(
                hemorrhage_dir, f"{column.replace(' ', '_')}_panel.png"
            )
            plt.savefig(output_file, dpi=300, bbox_inches='tight')
            plt.close()

# ----------------------------------------
# Run plotting
# ----------------------------------------
create_panel_plots_by_hemorrhage()


In [13]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Output path
output_file = "Hemorrhage_Occlusion_3x3_grid.png"

# Load categorization data
df = pd.read_excel('Experiment Tracker.xlsx', sheet_name=['T30 categorization', 'T60 categorization'])

group_10 = set(df['T30 categorization']['Group_10'].dropna())
group_20 = set(df['T30 categorization']['Group_20'].dropna())
group_30 = set(df['T30 categorization']['Group_30'].dropna())

full_occlusion = set(df['T60 categorization']['Full_Occlusion'].dropna())
partial_occlusion = set(df['T60 categorization']['Partial_Occlusion'].dropna())
no_occlusion = set(df['T60 categorization']['No_Occlusion'].dropna())

group_combinations = {
    "10": {"No": group_10.intersection(no_occlusion),
           "Partial": group_10.intersection(partial_occlusion),
           "Full": group_10.intersection(full_occlusion)},
    "20": {"No": group_20.intersection(no_occlusion),
           "Partial": group_20.intersection(partial_occlusion),
           "Full": group_20.intersection(full_occlusion)},
    "30": {"No": group_30.intersection(no_occlusion),
           "Partial": group_30.intersection(partial_occlusion),
           "Full": group_30.intersection(full_occlusion)}
}

# Load processed ERNE data
xls = pd.ExcelFile("Processed_Data_with_Vasopressin.xlsx")
erne_sheets = [s for s in xls.sheet_names if s.startswith("ERNE")]
data_dict = {}
for sheet in erne_sheets:
    df_sheet = xls.parse(sheet)
    new_sheet_name = sheet.replace("-", "")
    required_columns = [
        "Time", "Cumulative Vasopressin", "Plasmalyte", "Norepi",
        "Prox_Sys_PowerLab_PC", "Prox_Dia_PowerLab_PC", "Prox_Mean_PowerLab_PC",
        "Dist_Sys_PowerLab_PC", "Dist_Dia_PowerLab_PC", "Dist_Mean_PowerLab_PC",
        "Dist_Sys_LPF_PowerLab_PC", "Dist_Dia_LPF_PowerLab_PC", "Dist_Mean_LPF_PowerLab_PC"
    ]
    df_sheet = df_sheet[required_columns]
    df_sheet = df_sheet[df_sheet["Time"] <= 240]
    data_dict[new_sheet_name] = df_sheet

# Helper: compute mean ± SD
def calculate_mean_std(subjects, value_col, time_points):
    group_values = {}
    for sub in subjects:
        if sub in data_dict:
            df = data_dict[sub]
            df_filt = df[df["Time"].isin(time_points)]
            for _, row in df_filt.iterrows():
                t = row["Time"]
                group_values.setdefault(t, []).append(row[value_col])
    if not group_values:
        return [], [], []
    times = sorted(group_values.keys())
    means = [np.mean(group_values[t]) for t in times]
    stds = [np.std(group_values[t]) for t in times]
    return times, means, stds

time_points = [0,5,10,15,20,25,30,60,65,75,85,120,180,240]
columns_to_graph = ["Plasmalyte", "Cumulative Vasopressin", "Norepi"]  # Row order
occlusion_order = ["No", "Partial", "Full"]
hemorrhage_order = ["10", "20", "30"]
occlusion_styles = {
    "No": {"color": "black", "marker":"o"},
    "Partial": {"color":"blue", "marker":"o"},
    "Full": {"color":"red", "marker":"o"}
}

# Create figure with 3x3 grid
fig, axes = plt.subplots(len(columns_to_graph), len(hemorrhage_order), figsize=(18, 12), sharex=True)

for row_idx, column in enumerate(columns_to_graph):
    for col_idx, hemorrhage_level in enumerate(hemorrhage_order):
        ax = axes[row_idx, col_idx]
        
        # Plot one line per occlusion type
        for occlusion_type in occlusion_order:
            subjects = group_combinations[hemorrhage_level].get(occlusion_type, set())
            if not subjects:
                continue
            times, means, stds = calculate_mean_std(subjects, column, time_points)
            if times:
                style = occlusion_styles[occlusion_type]
                ax.plot(times, means, label=occlusion_type, color=style["color"], marker=style["marker"])
                ax.fill_between(times, np.array(means)-np.array(stds), np.array(means)+np.array(stds),
                                color=style["color"], alpha=0.2)
        
        if row_idx == 0:
            ax.set_title(f"{hemorrhage_level}% Hemorrhage", fontsize=14)
        if col_idx == 0:
            ax.set_ylabel(column, fontsize=14)
        ax.grid(True, linestyle='--', linewidth=0.5)

# Add single legend for occlusion types
fig.legend(occlusion_order, loc='upper right', bbox_to_anchor=(0.95, 0.95))
plt.tight_layout()
plt.savefig(output_file, dpi=300)
plt.close()


In [23]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ============================================================
# User Controls / Customization Section
# ============================================================

# Output path
output_file = "Hemorrhage_Occlusion_3x3_grid.png"

# Font and style controls
TITLE_FONTSIZE = 16       # TODO: edit
LABEL_FONTSIZE = 16       # TODO: edit
TICK_FONTSIZE = 16        # TODO: edit
LEGEND_FONTSIZE = 12      # TODO: edit

# Y-axis labels (include units)
# TODO: update units accordingly
y_axis_labels = {
    "Plasmalyte": "Plasmalyte (mL)",
    "Cumulative Vasopressin": "Cumulative Vasopressin (pg/mL)",
    "Norepi": "Norepinephrine (ng/kg/min)"
}

# ============================================================
# Data Loading Section
# ============================================================

# Load categorization data
df = pd.read_excel('Experiment Tracker.xlsx', sheet_name=['T30 categorization', 'T60 categorization'])

group_10 = set(df['T30 categorization']['Group_10'].dropna())
group_20 = set(df['T30 categorization']['Group_20'].dropna())
group_30 = set(df['T30 categorization']['Group_30'].dropna())

full_occlusion = set(df['T60 categorization']['Full_Occlusion'].dropna())
partial_occlusion = set(df['T60 categorization']['Partial_Occlusion'].dropna())
no_occlusion = set(df['T60 categorization']['No_Occlusion'].dropna())

group_combinations = {
    "10": {"No": group_10.intersection(no_occlusion),
           "Partial": group_10.intersection(partial_occlusion),
           "Full": group_10.intersection(full_occlusion)},
    "20": {"No": group_20.intersection(no_occlusion),
           "Partial": group_20.intersection(partial_occlusion),
           "Full": group_20.intersection(full_occlusion)},
    "30": {"No": group_30.intersection(no_occlusion),
           "Partial": group_30.intersection(partial_occlusion),
           "Full": group_30.intersection(full_occlusion)}
}

# Load processed ERNE data
xls = pd.ExcelFile("Processed_Data_with_Vasopressin.xlsx")
erne_sheets = [s for s in xls.sheet_names if s.startswith("ERNE")]
data_dict = {}
for sheet in erne_sheets:
    df_sheet = xls.parse(sheet)
    new_sheet_name = sheet.replace("-", "")
    required_columns = [
        "Time", "Cumulative Vasopressin", "Plasmalyte", "Norepi",
        "Prox_Sys_PowerLab_PC", "Prox_Dia_PowerLab_PC", "Prox_Mean_PowerLab_PC",
        "Dist_Sys_PowerLab_PC", "Dist_Dia_PowerLab_PC", "Dist_Mean_PowerLab_PC",
        "Dist_Sys_LPF_PowerLab_PC", "Dist_Dia_LPF_PowerLab_PC", "Dist_Mean_LPF_PowerLab_PC"
    ]
    df_sheet = df_sheet[required_columns]
    df_sheet = df_sheet[df_sheet["Time"] <= 240]
    data_dict[new_sheet_name] = df_sheet

# ============================================================
# Helper Functions
# ============================================================

def calculate_mean_std(subjects, value_col, time_points):
    """Compute mean ± SD for a given set of subjects and timepoints."""
    group_values = {}
    for sub in subjects:
        if sub in data_dict:
            df = data_dict[sub]
            df_filt = df[df["Time"].isin(time_points)]
            for _, row in df_filt.iterrows():
                t = row["Time"]
                group_values.setdefault(t, []).append(row[value_col])
    if not group_values:
        return [], [], []
    times = sorted(group_values.keys())
    means = [np.mean(group_values[t]) for t in times]
    stds = [np.std(group_values[t]) for t in times]
    return times, means, stds


# ============================================================
# Plotting Section
# ============================================================

time_points = [0, 5, 10, 15, 20, 25, 30, 60, 65, 75, 85, 120, 180, 240]
columns_to_graph = ["Plasmalyte", "Cumulative Vasopressin", "Norepi"]  # rows
hemorrhage_order = ["10", "20", "30"]  # columns
occlusion_order = ["No", "Partial", "Full"]

occlusion_styles = {
    "No": {"color": "black", "marker": "o"},
    "Partial": {"color": "blue", "marker": "o"},
    "Full": {"color": "red", "marker": "o"}
}

fig, axes = plt.subplots(len(columns_to_graph), len(hemorrhage_order),
                         figsize=(18, 12), sharex=True, sharey=False)

for row_idx, column in enumerate(columns_to_graph):
    for col_idx, hemorrhage_level in enumerate(hemorrhage_order):
        ax = axes[row_idx, col_idx]
        lines = []  # for legend handles

        for occlusion_type in occlusion_order:
            subjects = group_combinations[hemorrhage_level].get(occlusion_type, set())
            if not subjects:
                continue
            times, means, stds = calculate_mean_std(subjects, column, time_points)
            if times:
                style = occlusion_styles[occlusion_type]
                line, = ax.plot(times, means,
                                label=occlusion_type,
                                color=style["color"],
                                marker=style["marker"],
                                linewidth=2)
                ax.fill_between(times,
                                np.array(means) - np.array(stds),
                                np.array(means) + np.array(stds),
                                color=style["color"], alpha=0.2)
                lines.append(line)

        # Titles and labels
        if row_idx == 0:
            ax.set_title(f"{hemorrhage_level}% Hemorrhage", fontsize=TITLE_FONTSIZE)
        if col_idx == 0:
            ax.set_ylabel(y_axis_labels[column], fontsize=LABEL_FONTSIZE)

        # Grid and tick formatting
        ax.grid(True, linestyle="--", linewidth=0.5)
        ax.tick_params(axis='both', which='major', labelsize=TICK_FONTSIZE)

# Create a single, clean legend for occlusion types
handles = [plt.Line2D([0], [0], color=occlusion_styles[o]["color"],
                      marker=occlusion_styles[o]["marker"],
                      label=o, linewidth=2)
           for o in occlusion_order]
fig.legend(handles=handles,
           loc='upper right',
           bbox_to_anchor=(0.17, 0.95),
           fontsize=LEGEND_FONTSIZE,
           title="Occlusion Type",
           title_fontsize=LABEL_FONTSIZE)

plt.tight_layout(rect=[0, 0, 0.93, 1])  # leave space for legend
plt.savefig(output_file, dpi=300)
plt.close()


# ADDING HETASTARCH

In [ ]:
# import os
# import pandas as pd
# import numpy as np
# import matplotlib.pyplot as plt

# # ============================================================
# # User Controls / Customization Section
# # ============================================================

# output_file = "Hemorrhage_Occlusion_4x3_grid_with_Hetastarch.png"

# # Font and style controls
# TITLE_FONTSIZE = 16
# LABEL_FONTSIZE = 16
# TICK_FONTSIZE = 16
# LEGEND_FONTSIZE = 12

# # Y-axis labels (include units)
# y_axis_labels = {
#     "Plasmalyte": "Plasmalyte/weight (mL/kg)",
#     "Cumulative Vasopressin": "Cumulative Vasopressin (pg/mL)",
#     "Norepi": "Norepinephrine (ng/kg/min)",
#     "Hetastarch": "Hetastarch (mL)"
# }

# # ============================================================
# # Data Loading Section
# # ============================================================

# # Load categorization data
# df = pd.read_excel('Experiment Tracker.xlsx', sheet_name=['T30 categorization', 'T60 categorization'])

# group_10 = set(df['T30 categorization']['Group_10'].dropna())
# group_20 = set(df['T30 categorization']['Group_20'].dropna())
# group_30 = set(df['T30 categorization']['Group_30'].dropna())

# full_occlusion = set(df['T60 categorization']['Full_Occlusion'].dropna())
# partial_occlusion = set(df['T60 categorization']['Partial_Occlusion'].dropna())
# no_occlusion = set(df['T60 categorization']['No_Occlusion'].dropna())

# group_combinations = {
#     "10": {"No": group_10.intersection(no_occlusion),
#            "Partial": group_10.intersection(partial_occlusion),
#            "Full": group_10.intersection(full_occlusion)},
#     "20": {"No": group_20.intersection(no_occlusion),
#            "Partial": group_20.intersection(partial_occlusion),
#            "Full": group_20.intersection(full_occlusion)},
#     "30": {"No": group_30.intersection(no_occlusion),
#            "Partial": group_30.intersection(partial_occlusion),
#            "Full": group_30.intersection(full_occlusion)}
# }

# # Load processed ERNE data
# xls = pd.ExcelFile("Processed_Data_with_Vasopressin.xlsx")
# erne_sheets = [s for s in xls.sheet_names if s.startswith("ERNE")]
# data_dict = {}
# for sheet in erne_sheets:
#     df_sheet = xls.parse(sheet)
#     new_sheet_name = sheet.replace("-", "")
#     required_columns = [
#         "Time", "Cumulative Vasopressin", "Plasmalyte", "Norepi",
#         "Prox_Sys_PowerLab_PC", "Prox_Dia_PowerLab_PC", "Prox_Mean_PowerLab_PC",
#         "Dist_Sys_PowerLab_PC", "Dist_Dia_PowerLab_PC", "Dist_Mean_PowerLab_PC",
#         "Dist_Sys_LPF_PowerLab_PC", "Dist_Dia_LPF_PowerLab_PC", "Dist_Mean_LPF_PowerLab_PC"
#     ]
#     df_sheet = df_sheet[required_columns]
#     df_sheet = df_sheet[df_sheet["Time"] <= 240]
#     data_dict[new_sheet_name] = df_sheet

# # Load Hetastarch data
# hetastarch_df = pd.read_excel("Hetastarch Raw.xlsx")
# hetastarch_df.columns = ["Parent Folder", "Hetastarch (mL)"]
# hetastarch_data = dict(zip(hetastarch_df["Parent Folder"], hetastarch_df["Hetastarch (mL)"]))

# # ============================================================
# # Helper Functions
# # ============================================================

# def calculate_mean_std(subjects, value_col, time_points):
#     """Compute mean ± SD for a given set of subjects and timepoints."""
#     group_values = {}
#     for sub in subjects:
#         if sub in data_dict:
#             df = data_dict[sub]
#             df_filt = df[df["Time"].isin(time_points)]
#             for _, row in df_filt.iterrows():
#                 t = row["Time"]
#                 group_values.setdefault(t, []).append(row[value_col])
#     if not group_values:
#         return [], [], []
#     times = sorted(group_values.keys())
#     means = [np.mean(group_values[t]) for t in times]
#     stds = [np.std(group_values[t]) for t in times]
#     return times, means, stds


# # ============================================================
# # Plotting Section
# # ============================================================

# time_points = [0, 5, 10, 15, 20, 25, 30, 60, 65, 75, 85, 120, 180, 240]
# columns_to_graph = ["Plasmalyte", "Cumulative Vasopressin", "Norepi", "Hetastarch"]  # now includes Hetastarch
# hemorrhage_order = ["10", "20", "30"]
# occlusion_order = ["No", "Partial", "Full"]

# occlusion_styles = {
#     "No": {"color": "black", "marker": "o"},
#     "Partial": {"color": "blue", "marker": "o"},
#     "Full": {"color": "red", "marker": "o"}
# }

# fig, axes = plt.subplots(len(columns_to_graph), len(hemorrhage_order),
#                          figsize=(18, 16), sharex=False, sharey=False)

# for row_idx, column in enumerate(columns_to_graph):
#     for col_idx, hemorrhage_level in enumerate(hemorrhage_order):
#         ax = axes[row_idx, col_idx]
#         lines = []

#         if column != "Hetastarch":
#             # Time-series plotting for ERNE data
#             for occlusion_type in occlusion_order:
#                 subjects = group_combinations[hemorrhage_level].get(occlusion_type, set())
#                 if not subjects:
#                     continue
#                 times, means, stds = calculate_mean_std(subjects, column, time_points)
#                 if times:
#                     style = occlusion_styles[occlusion_type]
#                     line, = ax.plot(times, means,
#                                     label=occlusion_type,
#                                     color=style["color"],
#                                     marker=style["marker"],
#                                     linewidth=2)
#                     ax.fill_between(times,
#                                     np.array(means) - np.array(stds),
#                                     np.array(means) + np.array(stds),
#                                     color=style["color"], alpha=0.2)
#                     lines.append(line)
#         else:
#             # Boxplot row for Hetastarch
#             data_to_plot = []
#             colors = []
#             labels = []

#             for occlusion_type in occlusion_order:
#                 subjects = group_combinations[hemorrhage_level].get(occlusion_type, set())
#                 values = [hetastarch_data[sub] for sub in subjects if sub in hetastarch_data]
#                 if values:
#                     data_to_plot.append(values)
#                     colors.append(occlusion_styles[occlusion_type]["color"])
#                     labels.append(occlusion_type)

#             if data_to_plot:
#                 bplots = ax.boxplot(data_to_plot,
#                                     patch_artist=True,
#                                     labels=labels)
#                 for patch, color in zip(bplots['boxes'], colors):
#                     patch.set_facecolor(color)
#                     patch.set_alpha(0.4)

#         # Titles and labels
#         if row_idx == 0:
#             ax.set_title(f"{hemorrhage_level}% Hemorrhage", fontsize=TITLE_FONTSIZE)
#         if col_idx == 0:
#             ax.set_ylabel(y_axis_labels[column], fontsize=LABEL_FONTSIZE)

#         # Grid and ticks
#         ax.grid(True, linestyle="--", linewidth=0.5)
#         ax.tick_params(axis='both', which='major', labelsize=TICK_FONTSIZE)

# # Legend
# handles = [plt.Line2D([0], [0], color=occlusion_styles[o]["color"],
#                       marker=occlusion_styles[o]["marker"],
#                       label=o, linewidth=2)
#            for o in occlusion_order]
# fig.legend(handles=handles,
#            loc='upper right',
#            bbox_to_anchor=(0.17, 0.97),
#            fontsize=LEGEND_FONTSIZE,
#            title="Occlusion Type",
#            title_fontsize=LABEL_FONTSIZE)

# plt.tight_layout(rect=[0, 0, 0.93, 1])
# plt.savefig(output_file, dpi=300)
# plt.close()


In [6]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ============================================================
# User Controls / Customization Section
# ============================================================

output_file = "Hemorrhage_Occlusion_4x3_grid_with_Hetastarch.png"

# Font and style controls
TITLE_FONTSIZE = 16
LABEL_FONTSIZE = 16
TICK_FONTSIZE = 16
LEGEND_FONTSIZE = 12

# Y-axis labels (include units)
y_axis_labels = {
    "Plasmalyte": "Plasmalyte / weight \n (mL/kg)",
    "Cumulative Vasopressin": "Cumulative Vasopressin / weight \n (mU/kg)",
    "Norepi": "Norepinephrine / weight \n (mcg/kg)",
    "Hetastarch": "Hetastarch \n (mL)"
}

# ============================================================
# Data Loading Section
# ============================================================

# Load categorization data
df = pd.read_excel('Experiment Tracker.xlsx', sheet_name=['T30 categorization', 'T60 categorization'])

group_10 = set(df['T30 categorization']['Group_10'].dropna())
group_20 = set(df['T30 categorization']['Group_20'].dropna())
group_30 = set(df['T30 categorization']['Group_30'].dropna())

full_occlusion = set(df['T60 categorization']['Full_Occlusion'].dropna())
partial_occlusion = set(df['T60 categorization']['Partial_Occlusion'].dropna())
no_occlusion = set(df['T60 categorization']['No_Occlusion'].dropna())

group_combinations = {
    "10": {"No": group_10.intersection(no_occlusion),
           "Partial": group_10.intersection(partial_occlusion),
           "Full": group_10.intersection(full_occlusion)},
    "20": {"No": group_20.intersection(no_occlusion),
           "Partial": group_20.intersection(partial_occlusion),
           "Full": group_20.intersection(full_occlusion)},
    "30": {"No": group_30.intersection(no_occlusion),
           "Partial": group_30.intersection(partial_occlusion),
           "Full": group_30.intersection(full_occlusion)}
}

# Load processed ERNE data
xls = pd.ExcelFile("Processed_Data_with_Vasopressin.xlsx")
erne_sheets = [s for s in xls.sheet_names if s.startswith("ERNE")]
data_dict = {}
for sheet in erne_sheets:
    df_sheet = xls.parse(sheet)
    new_sheet_name = sheet.replace("-", "")
    required_columns = [
        "Time", "Cumulative Vasopressin", "Plasmalyte", "Norepi",
        "Prox_Sys_PowerLab_PC", "Prox_Dia_PowerLab_PC", "Prox_Mean_PowerLab_PC",
        "Dist_Sys_PowerLab_PC", "Dist_Dia_PowerLab_PC", "Dist_Mean_PowerLab_PC",
        "Dist_Sys_LPF_PowerLab_PC", "Dist_Dia_LPF_PowerLab_PC", "Dist_Mean_LPF_PowerLab_PC"
    ]
    df_sheet = df_sheet[required_columns]
    df_sheet = df_sheet[df_sheet["Time"] <= 240]
    data_dict[new_sheet_name] = df_sheet

# Load Hetastarch data
hetastarch_df = pd.read_excel("Hetastarch Raw.xlsx")
hetastarch_df.columns = ["Parent Folder", "Hetastarch (mL)"]
hetastarch_data = dict(zip(hetastarch_df["Parent Folder"], hetastarch_df["Hetastarch (mL)"]))

# ============================================================
# Helper Functions
# ============================================================

def calculate_mean_std(subjects, value_col, time_points):
    """Compute mean ± SD for a given set of subjects and timepoints."""
    group_values = {}
    for sub in subjects:
        if sub in data_dict:
            df = data_dict[sub]
            df_filt = df[df["Time"].isin(time_points)]
            for _, row in df_filt.iterrows():
                t = row["Time"]
                group_values.setdefault(t, []).append(row[value_col])
    if not group_values:
        return [], [], []
    times = sorted(group_values.keys())
    means = [np.mean(group_values[t]) for t in times]
    stds = [np.std(group_values[t]) for t in times]
    return times, means, stds

# ============================================================
# Plotting Section
# ============================================================

time_points = [0, 5, 10, 15, 20, 25, 30, 60, 65, 75, 85, 120, 180, 240]  # ERNE data points
columns_to_graph = ["Plasmalyte", "Cumulative Vasopressin", "Norepi", "Hetastarch"]
hemorrhage_order = ["10", "20", "30"]
occlusion_order = ["No", "Partial", "Full"]

occlusion_styles = {
    "No": {"color": "black", "marker": "o"},
    "Partial": {"color": "blue", "marker": "o"},
    "Full": {"color": "red", "marker": "o"}
}

# Compute y-limits for consistent axes per row
y_limits = {}
for column in columns_to_graph:
    all_values = []
    if column != "Hetastarch":
        for sub, df in data_dict.items():
            all_values.extend(df[column].values)
    else:
        all_values.extend(list(hetastarch_data.values()))
        all_values.append(0)  # include Time=30 value
    if all_values:
        y_limits[column] = (min(all_values) * 0.95, max(all_values) * 1.05)
    else:
        y_limits[column] = (0, 1)  # fallback

fig, axes = plt.subplots(len(columns_to_graph), len(hemorrhage_order),
                         figsize=(18, 16), sharex=False, sharey=False)

for row_idx, column in enumerate(columns_to_graph):
    for col_idx, hemorrhage_level in enumerate(hemorrhage_order):
        ax = axes[row_idx, col_idx]

        for occlusion_type in occlusion_order:
            subjects = group_combinations[hemorrhage_level].get(occlusion_type, set())
            if not subjects:
                continue

            if column != "Hetastarch":
                # Time-series plotting for ERNE data
                times, means, stds = calculate_mean_std(subjects, column, time_points)
            else:
                # Hetastarch line plot: 0 at 30, same value at 60+
                times = [30, 60, 120, 180, 240]
                values = [hetastarch_data[sub] for sub in subjects if sub in hetastarch_data]
                if not values:
                    continue
                mean_val = np.mean(values)
                std_val = np.std(values)
                means = [0] + [mean_val] * 4
                stds = [0] + [std_val] * 4

            # Plot line + fill between
            style = occlusion_styles[occlusion_type]
            line, = ax.plot(times, means,
                            label=occlusion_type,
                            color=style["color"],
                            marker=style["marker"],
                            linewidth=2)
            ax.fill_between(times,
                            np.array(means) - np.array(stds),
                            np.array(means) + np.array(stds),
                            color=style["color"], alpha=0.2)

        # Titles and labels
        if row_idx == 0:
            ax.set_title(f"{hemorrhage_level}% Hemorrhage", fontsize=TITLE_FONTSIZE)
        if col_idx == 0:
            ax.set_ylabel(y_axis_labels[column], fontsize=LABEL_FONTSIZE)

        # Grid, ticks, and consistent y-axis
        ax.grid(True, linestyle="--", linewidth=0.5)
        ax.tick_params(axis='both', which='major', labelsize=TICK_FONTSIZE)
        ax.set_ylim(y_limits[column])

# Legend
handles = [plt.Line2D([0], [0], color=occlusion_styles[o]["color"],
                      marker=occlusion_styles[o]["marker"],
                      label=o, linewidth=2)
           for o in occlusion_order]
fig.legend(handles=handles,
           loc='upper right',
           bbox_to_anchor=(0.20, 0.97),
           fontsize=LEGEND_FONTSIZE,
           title="Occlusion Type",
           title_fontsize=LABEL_FONTSIZE)

plt.tight_layout(rect=[0, 0, 0.93, 1])
plt.savefig(output_file, dpi=300)
plt.close()


In [13]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ============================================================
# User Controls / Customization Section
# ============================================================

output_file = "Hemorrhage_Occlusion_4x3_grid_with_Hetastarch_Violin_YSynced.png"
hetastarch_output_file = "Processed_Hetastarch_Data.xlsx"

# Font and style controls
TITLE_FONTSIZE = 16
LABEL_FONTSIZE = 16
TICK_FONTSIZE = 16
LEGEND_FONTSIZE = 12

# Y-axis labels (include units)
y_axis_labels = {
    "Plasmalyte": "Plasmalyte (mL/kg)",
    "Cumulative Vasopressin": "Vasopressin (mU/kg)",
    "Norepi": "Norepinephrine (mcg/kg)",
    "Hetastarch": "Hetastarch (mL/kg)"
}

# ============================================================
# Data Loading Section
# ============================================================

# Load categorization data
df = pd.read_excel('Experiment Tracker.xlsx', sheet_name=['T30 categorization', 'T60 categorization'])

group_10 = set(df['T30 categorization']['Group_10'].dropna())
group_20 = set(df['T30 categorization']['Group_20'].dropna())
group_30 = set(df['T30 categorization']['Group_30'].dropna())

full_occlusion = set(df['T60 categorization']['Full_Occlusion'].dropna())
partial_occlusion = set(df['T60 categorization']['Partial_Occlusion'].dropna())
no_occlusion = set(df['T60 categorization']['No_Occlusion'].dropna())

group_combinations = {
    "10": {"No": group_10.intersection(no_occlusion),
           "Partial": group_10.intersection(partial_occlusion),
           "Full": group_10.intersection(full_occlusion)},
    "20": {"No": group_20.intersection(no_occlusion),
           "Partial": group_20.intersection(partial_occlusion),
           "Full": group_20.intersection(full_occlusion)},
    "30": {"No": group_30.intersection(no_occlusion),
           "Partial": group_30.intersection(partial_occlusion),
           "Full": group_30.intersection(full_occlusion)}
}

# Load processed ERNE data
xls = pd.ExcelFile("Processed_Data_with_Vasopressin.xlsx")
erne_sheets = [s for s in xls.sheet_names if s.startswith("ERNE")]
data_dict = {}
for sheet in erne_sheets:
    df_sheet = xls.parse(sheet)
    new_sheet_name = sheet.replace("-", "")
    required_columns = [
        "Time", "Cumulative Vasopressin", "Plasmalyte", "Norepi",
        "Prox_Sys_PowerLab_PC", "Prox_Dia_PowerLab_PC", "Prox_Mean_PowerLab_PC",
        "Dist_Sys_PowerLab_PC", "Dist_Dia_PowerLab_PC", "Dist_Mean_PowerLab_PC",
        "Dist_Sys_LPF_PowerLab_PC", "Dist_Dia_LPF_PowerLab_PC", "Dist_Mean_LPF_PowerLab_PC"
    ]
    df_sheet = df_sheet[required_columns]
    df_sheet = df_sheet[df_sheet["Time"] <= 240]
    data_dict[new_sheet_name] = df_sheet

# Load Hetastarch data
hetastarch_df = pd.read_excel("Hetastarch Raw.xlsx")
hetastarch_df.columns = ["Parent Folder", "Hetastarch (mL)"]

# Load Weights data and merge to normalize
weights_df = pd.read_excel("Weights.xlsx")
weights_df.columns = ["Parent Folder", "Weight"]

# Merge Hetastarch and Weights
merged_het = pd.merge(hetastarch_df, weights_df, on="Parent Folder", how="inner")
merged_het["Hetastarch / Weight (mL/kg)"] = merged_het["Hetastarch (mL)"] / merged_het["Weight"]
hetastarch_data = dict(zip(merged_het["Parent Folder"], merged_het["Hetastarch / Weight (mL/kg)"]))

# Add Occlusion Group and Hemorrhage Level columns for output file
occlusion_labels = []
hemorrhage_labels = []
for subject in merged_het["Parent Folder"]:
    found_hemorrhage, found_occlusion = None, None
    for h_level, occ_dict in group_combinations.items():
        for occ_type, subj_list in occ_dict.items():
            if subject in subj_list:
                found_hemorrhage = h_level
                found_occlusion = occ_type
                break
        if found_hemorrhage:
            break
    occlusion_labels.append(found_occlusion)
    hemorrhage_labels.append(found_hemorrhage)

merged_het["Occlusion Group"] = occlusion_labels
merged_het["Hemorrhage Level"] = hemorrhage_labels
merged_het.to_excel(hetastarch_output_file, index=False)

# ============================================================
# Helper Functions
# ============================================================

def calculate_mean_std(subjects, value_col, time_points):
    """Compute mean ± SD for a given set of subjects and timepoints."""
    group_values = {}
    for sub in subjects:
        if sub in data_dict:
            df = data_dict[sub]
            df_filt = df[df["Time"].isin(time_points)]
            for _, row in df_filt.iterrows():
                t = row["Time"]
                group_values.setdefault(t, []).append(row[value_col])
    if not group_values:
        return [], [], []
    times = sorted(group_values.keys())
    means = [np.mean(group_values[t]) for t in times]
    stds = [np.std(group_values[t]) for t in times]
    return times, means, stds

# ============================================================
# Plotting Section
# ============================================================

time_points = [0, 5, 10, 15, 20, 25, 30, 60, 65, 75, 85, 120, 180, 240]
columns_to_graph = ["Plasmalyte", "Cumulative Vasopressin", "Norepi", "Hetastarch"]
hemorrhage_order = ["10", "20", "30"]
occlusion_order = ["No", "Partial", "Full"]

occlusion_styles = {
    "No": {"color": "black", "marker": "o"},
    "Partial": {"color": "blue", "marker": "o"},
    "Full": {"color": "red", "marker": "o"}
}

fig, axes = plt.subplots(len(columns_to_graph), len(hemorrhage_order),
                         figsize=(18, 16), sharex=False, sharey=False)

# For consistent y-limits across each row
row_y_limits = {c: [np.inf, -np.inf] for c in columns_to_graph}
plot_data = {}

# --- First pass: gather data to compute y-axis limits ---
for row_idx, column in enumerate(columns_to_graph):
    plot_data[column] = {}
    for hemorrhage_level in hemorrhage_order:
        plot_data[column][hemorrhage_level] = {}
        for occlusion_type in occlusion_order:
            subjects = group_combinations[hemorrhage_level].get(occlusion_type, set())
            if column != "Hetastarch":
                times, means, stds = calculate_mean_std(subjects, column, time_points)
                if means:
                    ymin, ymax = min(means) - max(stds, default=0), max(means) + max(stds, default=0)
                    row_y_limits[column][0] = min(row_y_limits[column][0], ymin)
                    row_y_limits[column][1] = max(row_y_limits[column][1], ymax)
                plot_data[column][hemorrhage_level][occlusion_type] = (times, means, stds)
            else:
                # Store Hetastarch data for later violin plot
                values = [hetastarch_data[sub] for sub in subjects if sub in hetastarch_data]
                if values:
                    ymin, ymax = min(values), max(values)
                    row_y_limits[column][0] = min(row_y_limits[column][0], ymin)
                    row_y_limits[column][1] = max(row_y_limits[column][1], ymax)
                plot_data[column][hemorrhage_level][occlusion_type] = values

# --- Second pass: create plots ---
for row_idx, column in enumerate(columns_to_graph):
    for col_idx, hemorrhage_level in enumerate(hemorrhage_order):
        ax = axes[row_idx, col_idx]

        if column != "Hetastarch":
            # Line plots for time-series variables
            for occlusion_type in occlusion_order:
                times, means, stds = plot_data[column][hemorrhage_level][occlusion_type]
                if not times:
                    continue
                style = occlusion_styles[occlusion_type]
                ax.plot(times, means, label=occlusion_type,
                        color=style["color"], marker=style["marker"], linewidth=2)
                ax.fill_between(times, np.array(means) - np.array(stds),
                                np.array(means) + np.array(stds),
                                color=style["color"], alpha=0.2)

            ax.set_xticks(np.arange(0, 241, 30))
            ax.set_xlim(0, 240)
            ax.set_ylim(row_y_limits[column])

        else:
            # Violin plots for Hetastarch
            data_to_plot = []
            colors = []
            labels = []

            for occlusion_type in occlusion_order:
                values = plot_data[column][hemorrhage_level][occlusion_type]
                if values:
                    data_to_plot.append(values)
                    colors.append(occlusion_styles[occlusion_type]["color"])
                    labels.append(occlusion_type)

            if data_to_plot:
                parts = ax.violinplot(data_to_plot, showmeans=False, showmedians=True, showextrema=False)
                for pc, color in zip(parts['bodies'], colors):
                    pc.set_facecolor(color)
                    pc.set_alpha(0.4)
                    pc.set_edgecolor('black')
                    pc.set_linewidth(1.2)
                if 'cmedians' in parts:
                    parts['cmedians'].set_color('black')
                    parts['cmedians'].set_linewidth(2)

                ax.set_xticks(np.arange(1, len(labels) + 1))
                ax.set_xticklabels(labels, fontsize=TICK_FONTSIZE)
                ax.set_xlim(0.5, len(labels) + 0.5)
                ax.set_ylim(row_y_limits[column])

        if row_idx == 0:
            ax.set_title(f"{hemorrhage_level}% Hemorrhage", fontsize=TITLE_FONTSIZE)
        if col_idx == 0:
            ax.set_ylabel(y_axis_labels[column], fontsize=LABEL_FONTSIZE)

        ax.grid(True, linestyle="--", linewidth=0.5)
        ax.tick_params(axis='both', which='major', labelsize=TICK_FONTSIZE)

# Legend
handles = [plt.Line2D([0], [0], color=occlusion_styles[o]["color"],
                      marker=occlusion_styles[o]["marker"],
                      label=o, linewidth=2)
           for o in occlusion_order]
fig.legend(handles=handles,
           loc='upper right',
           bbox_to_anchor=(0.17, 0.97),
           fontsize=LEGEND_FONTSIZE,
           title="Occlusion Type",
           title_fontsize=LABEL_FONTSIZE)

plt.tight_layout(rect=[0, 0, 0.93, 1])
plt.savefig(output_file, dpi=300)
plt.close()



In [14]:
import os
import pandas as pd
import numpy as np

# ============================================================
# Load Data
# ============================================================

# Load categorization data
df = pd.read_excel('Experiment Tracker.xlsx', sheet_name=['T30 categorization', 'T60 categorization'])

group_10 = set(df['T30 categorization']['Group_10'].dropna())
group_20 = set(df['T30 categorization']['Group_20'].dropna())
group_30 = set(df['T30 categorization']['Group_30'].dropna())

full_occlusion = set(df['T60 categorization']['Full_Occlusion'].dropna())
partial_occlusion = set(df['T60 categorization']['Partial_Occlusion'].dropna())
no_occlusion = set(df['T60 categorization']['No_Occlusion'].dropna())

group_combinations = {
    "10": {"No": group_10.intersection(no_occlusion),
           "Partial": group_10.intersection(partial_occlusion),
           "Full": group_10.intersection(full_occlusion)},
    "20": {"No": group_20.intersection(no_occlusion),
           "Partial": group_20.intersection(partial_occlusion),
           "Full": group_20.intersection(full_occlusion)},
    "30": {"No": group_30.intersection(no_occlusion),
           "Partial": group_30.intersection(partial_occlusion),
           "Full": group_30.intersection(full_occlusion)}
}

# Load processed ERNE data
xls = pd.ExcelFile("Processed_Data_with_Vasopressin.xlsx")
erne_sheets = [s for s in xls.sheet_names if s.startswith("ERNE")]
data_dict = {}
for sheet in erne_sheets:
    df_sheet = xls.parse(sheet)
    new_sheet_name = sheet.replace("-", "")
    df_sheet = df_sheet[df_sheet["Time"] <= 240]
    data_dict[new_sheet_name] = df_sheet

# Load Hetastarch data (already normalized)
hetastarch_df = pd.read_excel("Hetastarch Raw.xlsx")
hetastarch_df.columns = ["Parent Folder", "Hetastarch (mL)"]

weights_df = pd.read_excel("Weights.xlsx")
weights_df.columns = ["Parent Folder", "Weight"]

merged_het = pd.merge(hetastarch_df, weights_df, on="Parent Folder", how="inner")
merged_het["Hetastarch (mL/kg)"] = merged_het["Hetastarch (mL)"] / merged_het["Weight"]
hetastarch_data = dict(zip(merged_het["Parent Folder"], merged_het["Hetastarch (mL/kg)"]))

# ============================================================
# Helper function for mean ± std at time 240
# ============================================================

def mean_std(values):
    """Return formatted mean ± std string."""
    if len(values) == 0:
        return "-"
    return f"{np.mean(values):.2f} ± {np.std(values):.2f}"

def summarize_at_time(subjects, value_col):
    """Compute values at Time = 240 for the selected subjects."""
    vals = []
    for sub in subjects:
        if sub in data_dict:
            df = data_dict[sub]
            val = df.loc[df["Time"] == 240, value_col]
            if not val.empty:
                vals.append(val.values[0])
    return vals

# ============================================================
# Generate summary tables
# ============================================================

columns_to_summarize = [
    "Plasmalyte",
    "Cumulative Vasopressin",
    "Norepi",
    "Hetastarch (mL/kg)"
]

tables = {}

for hemorrhage_level in ["10", "20", "30"]:
    rows = []
    for param in columns_to_summarize:
        row_data = {}
        for occlusion_type in ["No", "Partial", "Full"]:
            subjects = group_combinations[hemorrhage_level].get(occlusion_type, set())

            if param == "Hetastarch (mL/kg)":
                vals = [hetastarch_data[sub] for sub in subjects if sub in hetastarch_data]
            else:
                vals = summarize_at_time(subjects, param)

            row_data[occlusion_type] = mean_std(vals)

        rows.append(row_data)

    df_table = pd.DataFrame(rows, index=columns_to_summarize)
    tables[hemorrhage_level] = df_table

# ============================================================
# Display or save the results
# ============================================================

for level, table in tables.items():
    print(f"\n===== Table: {level}% Hemorrhage =====")
    print(table)
    # Optionally, save each table
    table.to_excel(f"Summary_Table_{level}pct_Hemorrhage.xlsx")



===== Table: 10% Hemorrhage =====
                                  No        Partial           Full
Plasmalyte              40.74 ± 6.75  43.32 ± 16.18  61.15 ± 13.50
Cumulative Vasopressin  60.55 ± 5.11  27.20 ± 29.29   87.20 ± 4.27
Norepi                  33.06 ± 2.17  26.94 ± 21.12  83.55 ± 26.64
Hetastarch (mL/kg)       4.65 ± 1.37    2.64 ± 0.36              -

===== Table: 20% Hemorrhage =====
                                   No        Partial            Full
Plasmalyte               29.87 ± 6.26   30.90 ± 9.15   84.03 ± 14.05
Cumulative Vasopressin  20.91 ± 11.27  31.22 ± 24.10    86.66 ± 2.09
Norepi                   16.27 ± 2.65  19.47 ± 11.81  157.62 ± 41.38
Hetastarch (mL/kg)        4.00 ± 1.41    5.50 ± 3.20               -

===== Table: 30% Hemorrhage =====
                                   No        Partial          Full
Plasmalyte               28.19 ± 3.18   45.86 ± 7.50  71.16 ± 4.52
Cumulative Vasopressin  24.14 ± 24.14  55.42 ± 19.59  76.22 ± 0.62
Norepi        

In [20]:
import os
import re
import pandas as pd
import numpy as np

# ============================================================
# Load Data (same logic as your reference)
# ============================================================

# Load categorization data
df = pd.read_excel('Experiment Tracker.xlsx', sheet_name=['T30 categorization', 'T60 categorization'])

group_10 = set(df['T30 categorization']['Group_10'].dropna())
group_20 = set(df['T30 categorization']['Group_20'].dropna())
group_30 = set(df['T30 categorization']['Group_30'].dropna())

full_occlusion = set(df['T60 categorization']['Full_Occlusion'].dropna())
partial_occlusion = set(df['T60 categorization']['Partial_Occlusion'].dropna())
no_occlusion = set(df['T60 categorization']['No_Occlusion'].dropna())

group_combinations = {
    "10": {"No": group_10.intersection(no_occlusion),
           "Partial": group_10.intersection(partial_occlusion),
           "Full": group_10.intersection(full_occlusion)},
    "20": {"No": group_20.intersection(no_occlusion),
           "Partial": group_20.intersection(partial_occlusion),
           "Full": group_20.intersection(full_occlusion)},
    "30": {"No": group_30.intersection(no_occlusion),
           "Partial": group_30.intersection(partial_occlusion),
           "Full": group_30.intersection(full_occlusion)}
}

# Load processed ERNE data (same parsing as your reference)
xls = pd.ExcelFile("Processed_Data_with_Vasopressin.xlsx")
erne_sheets = [s for s in xls.sheet_names if s.startswith("ERNE")]
data_dict = {}
for sheet in erne_sheets:
    df_sheet = xls.parse(sheet)
    new_sheet_name = sheet.replace("-", "")
    # Keep rows up to 240 (as in your reference)
    df_sheet = df_sheet[df_sheet["Time"] <= 240]
    data_dict[new_sheet_name] = df_sheet

# Load Hetastarch data and normalize by weight (same as reference)
hetastarch_df = pd.read_excel("Hetastarch Raw.xlsx")
hetastarch_df.columns = ["Parent Folder", "Hetastarch (mL)"]

weights_df = pd.read_excel("Weights.xlsx")
weights_df.columns = ["Parent Folder", "Weight"]

merged_het = pd.merge(hetastarch_df, weights_df, on="Parent Folder", how="inner")
merged_het["Hetastarch (mL/kg)"] = merged_het["Hetastarch (mL)"] / merged_het["Weight"]
hetastarch_data = dict(zip(merged_het["Parent Folder"], merged_het["Hetastarch (mL/kg)"]))

# ============================================================
# Parameters / timepoints
# ============================================================

# Parameters to export (match your previous names)
columns_to_extract = [
    "Plasmalyte",
    "Cumulative Vasopressin",
    "Norepi",
    "Hetastarch (mL/kg)"
]

# Timepoints: hetastarch only at 60, others at 85,120,180,240
timepoints_non_het = [85, 120, 180, 240]
timepoints_het = [60]

# Output file
output_file = "Raw_Values_by_Parameter_and_Time.xlsx"

# ============================================================
# Helper: gather raw rows for a parameter
# ============================================================

def gather_rows_for_param(param_name):
    """
    Returns a list of dicts:
    Parent Folder | Hemorrhage Level | Occlusion Type | Time | Value
    """
    rows = []

    # iterate hemorrhage and occlusion combos as before
    for hemorrhage_level in ["10", "20", "30"]:
        for occlusion_type in ["No", "Partial", "Full"]:
            subjects = group_combinations[hemorrhage_level].get(occlusion_type, set())
            if not subjects:
                continue

            if param_name == "Hetastarch (mL/kg)":
                # Hetastarch: only timepoint 60, value from merged_het (normalized)
                for subj in subjects:
                    if subj in hetastarch_data:
                        rows.append({
                            "Parent Folder": subj,
                            "Hemorrhage Level": hemorrhage_level,
                            "Occlusion Type": occlusion_type,
                            "Time": 60,
                            "Value": hetastarch_data[subj]
                        })
            else:
                # Non-hetastarch: read from data_dict at times 85,120,180,240
                for subj in subjects:
                    if subj not in data_dict:
                        continue
                    df_sub = data_dict[subj]
                    for t in timepoints_non_het:
                        # find row(s) where Time == t
                        vals = df_sub.loc[df_sub["Time"] == t, param_name]
                        if not vals.empty:
                            # If multiple entries, iterate through them (but likely one)
                            for v in vals.values:
                                rows.append({
                                    "Parent Folder": subj,
                                    "Hemorrhage Level": hemorrhage_level,
                                    "Occlusion Type": occlusion_type,
                                    "Time": int(t),
                                    "Value": v
                                })
    return rows

# ============================================================
# Build dictionary of dataframes and write Excel
# ============================================================

def clean_sheet_name(name):
    safe = re.sub(r'[\\/*?:\[\]]', '_', name)
    return safe[:31]  # Excel sheet limit

with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
    for param in columns_to_extract:
        rows = gather_rows_for_param(param)
        if not rows:
            # create empty DF with standard columns to avoid missing sheet (optional)
            df_out = pd.DataFrame(columns=["Parent Folder", "Hemorrhage Level", "Occlusion Type", "Time", "Value"])
        else:
            df_out = pd.DataFrame(rows)
        sheet_name = clean_sheet_name(param)
        df_out.to_excel(writer, sheet_name=sheet_name, index=False)

print(f"✅ Exported raw values to '{output_file}' with sheets:")
for p in columns_to_extract:
    print("  -", clean_sheet_name(p))


✅ Exported raw values to 'Raw_Values_by_Parameter_and_Time.xlsx' with sheets:
  - Plasmalyte
  - Cumulative Vasopressin
  - Norepi
  - Hetastarch (mL_kg)


In [21]:
import os
import re
import pandas as pd
import numpy as np

# ============================================================
# Load Data (same logic as your reference)
# ============================================================

# Load categorization data
df = pd.read_excel('Experiment Tracker.xlsx', sheet_name=['T30 categorization', 'T60 categorization'])

group_10 = set(df['T30 categorization']['Group_10'].dropna())
group_20 = set(df['T30 categorization']['Group_20'].dropna())
group_30 = set(df['T30 categorization']['Group_30'].dropna())

full_occlusion = set(df['T60 categorization']['Full_Occlusion'].dropna())
partial_occlusion = set(df['T60 categorization']['Partial_Occlusion'].dropna())
no_occlusion = set(df['T60 categorization']['No_Occlusion'].dropna())

group_combinations = {
    "10": {"No": group_10.intersection(no_occlusion),
           "Partial": group_10.intersection(partial_occlusion),
           "Full": group_10.intersection(full_occlusion)},
    "20": {"No": group_20.intersection(no_occlusion),
           "Partial": group_20.intersection(partial_occlusion),
           "Full": group_20.intersection(full_occlusion)},
    "30": {"No": group_30.intersection(no_occlusion),
           "Partial": group_30.intersection(partial_occlusion),
           "Full": group_30.intersection(full_occlusion)}
}

# Load processed ERNE data (same parsing as your reference)
xls = pd.ExcelFile("Processed_Data_with_Vasopressin.xlsx")
erne_sheets = [s for s in xls.sheet_names if s.startswith("ERNE")]
data_dict = {}
for sheet in erne_sheets:
    df_sheet = xls.parse(sheet)
    new_sheet_name = sheet.replace("-", "")
    # Keep rows up to 240 (as in your reference)
    df_sheet = df_sheet[df_sheet["Time"] <= 240]
    data_dict[new_sheet_name] = df_sheet

# Load Hetastarch data and normalize by weight (same as reference)
hetastarch_df = pd.read_excel("Hetastarch Raw.xlsx")
hetastarch_df.columns = ["Parent Folder", "Hetastarch (mL)"]

weights_df = pd.read_excel("Weights.xlsx")
weights_df.columns = ["Parent Folder", "Weight"]

merged_het = pd.merge(hetastarch_df, weights_df, on="Parent Folder", how="inner")
merged_het["Hetastarch (mL/kg)"] = merged_het["Hetastarch (mL)"] / merged_het["Weight"]
hetastarch_data = dict(zip(merged_het["Parent Folder"], merged_het["Hetastarch (mL/kg)"]))

# ============================================================
# Parameters / timepoints
# ============================================================

# Parameters to export (match your previous names)
columns_to_extract = [
    "Plasmalyte",
    "Cumulative Vasopressin",
    "Norepi",
    "Hetastarch (mL/kg)"
]

# Timepoints: hetastarch only at 60, others at 85,120,180,240
timepoints_non_het = [85, 120, 180, 240]
timepoints_het = [60]

# Output file
output_file = "Raw_Values_by_Parameter_and_Time.xlsx"

# ============================================================
# Helper: gather raw rows for a parameter
# ============================================================

def gather_rows_for_param(param_name):
    """
    Returns a list of dicts:
    Parent Folder | Hemorrhage Level | Occlusion Type | Time | Value
    """
    rows = []

    # iterate hemorrhage and occlusion combos as before
    for hemorrhage_level in ["10", "20", "30"]:
        for occlusion_type in ["No", "Partial", "Full"]:
            subjects = group_combinations[hemorrhage_level].get(occlusion_type, set())
            if not subjects:
                continue

            if param_name == "Hetastarch (mL/kg)":
                # Hetastarch: only timepoint 60, value from merged_het (normalized)
                for subj in subjects:
                    if subj in hetastarch_data:
                        rows.append({
                            "Parent Folder": subj,
                            "Hemorrhage Level": hemorrhage_level,
                            "Occlusion Type": occlusion_type,
                            "Time": 60,
                            "Value": hetastarch_data[subj]
                        })
            else:
                # Non-hetastarch: read from data_dict at times 85,120,180,240
                for subj in subjects:
                    if subj not in data_dict:
                        continue
                    df_sub = data_dict[subj]
                    for t in timepoints_non_het:
                        # find row(s) where Time == t
                        vals = df_sub.loc[df_sub["Time"] == t, param_name]
                        if not vals.empty:
                            # If multiple entries, iterate through them (but likely one)
                            for v in vals.values:
                                rows.append({
                                    "Parent Folder": subj,
                                    "Hemorrhage Level": hemorrhage_level,
                                    "Occlusion Type": occlusion_type,
                                    "Time": int(t),
                                    "Value": v
                                })
    return rows

# ============================================================
# Build dictionary of dataframes and write Excel
# ============================================================

def clean_sheet_name(name):
    safe = re.sub(r'[\\/*?:\[\]]', '_', name)
    return safe[:31]  # Excel sheet limit

with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
    for param in columns_to_extract:
        rows = gather_rows_for_param(param)
        if not rows:
            # create empty DF with standard columns to avoid missing sheet (optional)
            df_out = pd.DataFrame(columns=["Parent Folder", "Hemorrhage Level", "Occlusion Type", "Time", "Value"])
        else:
            df_out = pd.DataFrame(rows)
        sheet_name = clean_sheet_name(param)
        df_out.to_excel(writer, sheet_name=sheet_name, index=False)

print(f"✅ Exported raw values to '{output_file}' with sheets:")
for p in columns_to_extract:
    print("  -", clean_sheet_name(p))


✅ Exported raw values to 'Raw_Values_by_Parameter_and_Time.xlsx' with sheets:
  - Plasmalyte
  - Cumulative Vasopressin
  - Norepi
  - Hetastarch (mL_kg)
